# BindCraft multi-template — design run

Hallucinates **one** binder sequence against ***N*** target structures at once.
Every template carries a strategy flag, so the same sequence can be pulled toward
some structures and pushed away from others:

- `target`: the binder must bind this structure.
- `detarget`: the binder must still *fold*, but it is penalised for forming a
  confident interface with this structure.

That covers both directions of the usual problem. Design for cross-reactivity
(bind human **and** mouse PD-L1, as configured here, where both templates are
`target`), or design for selectivity (bind one paralogue, ignore its close
relative).

---

## How it works

BindCraft designs by *hallucination*: there is no starting sequence. The binder
begins as free parameters, one probability distribution over the 20 amino acids at
each position, and AlphaFold2 is run as a differentiable function of them. Fold the
binder against the target, score the result, push the distributions in whichever
direction lowers the score, repeat. Only the sequence is ever updated; the
AlphaFold2 weights never change.

Across four stages that design is annealed from a blur into one real sequence:
free-floating logits first, then a sharpening softmax, then a forced commitment to a
single amino acid per position, and finally a discrete search over point mutations
(disabled in this run). Between stages the trajectory is gated on binder pLDDT, and
one that fails a gate is abandoned and logged to `failure_csv_*`.

**What multi-template changes.** A normal step is one fold and one update. Here
every step folds the binder against *each* template in turn, collects one gradient
per template, and combines them into a **single** update. The sequence is therefore
never an average of two separate designs: every step already moves in the joint
direction. How those gradients are combined (per-template normalisation, weighting,
an optional tilt toward whichever template is currently doing worst) is set in
**section 7b** and applies to stages 1-3; stage 4 averages the templates the
original way.

---

## Targeting vs. detargeting

Before each template's pass, `step_multi` rewrites the loss weights by multiplying
the base `weights_*` by that template's `detargeting_scaling_weights_*`. With the
patched settings this notebook loads:

| term | what it measures | `target` | `detarget` |
|---|---|---|---|
| `plddt` | binder fold confidence | 0.1 | × 1 |
| `pae_intra`, `con_intra` | binder's internal geometry | 0.4, 1.0 | × 1 |
| `pae_inter`, `con_inter` | binder↔target interface | 0.1, 1.5 | **× 0** |
| `iptm` | interface confidence | 0.1 | **× -1** |

So a `detarget` template says *"fold well, and do not form a confident interface
with me"*. The push comes from the `iptm` term alone. The loss `add_i_ptm_loss`
registers is `1 - i_pTM`, so flipping its weight to `-0.1` makes the optimiser
*maximise* `1 - i_pTM`, driving interface confidence down. The contact and PAE
interface terms stay at 0, so contacts themselves are neither rewarded nor
penalised; only the confidence of the interface is. Note this is live only because
`use_i_ptm_loss` is `true`; with that flag off the term is never registered and the
`-1` does nothing.

---

## Where the template swap happens

In the **Multi ColabDesign Utils** cell of section 7. One ColabDesign model is
built and `prep_inputs()` is called on it once per template, each call overwriting
the last, so each template's state is deep-copied out as it is prepped, at the top
of `binder_hallucination_multi()`. Every per-template pass then restores two things
onto the shared model:

- `model._inputs['batch']`, the target structure
- `model.opt["hotspot"]`, the residues being targeted

in `step_multi()` (stages 1-3), `design_mcmc_multi()` and
`design_semigreedy_multi()`. Everything else `prep_inputs()` wrote stays frozen at
the last-prepped template's value, which is why all templates must share the same
target residue count. This can be overcome by padding (dummy entries for the shorter target).

---

## After the trajectory

Standard BindCraft from here: MPNN sequence redesign, AF2 re-prediction, PyRosetta
interface scoring, filters. A design is accepted only if it passes *every*
template's filters. Sections 10-11 then re-fold the accepted binders against each
template **from sequence alone**

## Layout

Sections 0-5 define, validate and visualise the target. Sections 6-7b set up the
runtime and the aggregation knobs. Sections 8-9 run the design. Sections 10-11
re-predict and compare.

## Running this notebook

This notebook lives in the **original BindCraft** repository and must be run from a
checkout of it, with the notebook in `<repo>/notebooks/`. It runs the original,
**PyRosetta-based** BindCraft pipeline (interface scoring and relaxation via
PyRosetta), so the environment must have PyRosetta installed and the `DAlphaBall.gcc`
binary present.

- Original BindCraft (Martin Pacesa): https://github.com/martinpacesa/BindCraft
- FreeBindCraft (PyRosetta-free fork): https://github.com/cytokineking/FreeBindCraft

**Credits.** BindCraft is developed by the BindCraft development
team (https://github.com/martinpacesa/BindCraft). Please cite BindCraft if you use
this notebook. Special thanks to Paul Kittner and Casper Goverde for developing important bits of the code.

All paths are resolved **relative to the repo root** (the parent of `notebooks/`). The repo root is expected to contain:
`functions/` (the package plus the `dssp` / `DAlphaBall.gcc` binaries), `params/`
(AlphaFold2 weights, `params_*.npz`), `Inputs/` (your target PDBs), and
`settings_filters/`.

**You still need to adjust settings to your run:**
- Sections 1-2 (`GENERAL`, `TEMPLATES`): your target, templates and input PDBs.
- Make sure the AlphaFold2 weights are under `<repo>/params/`. If your weights or
  binaries live elsewhere, point `AF_PARAMS_DIR` / `DSSP` / `DALPHABALL` in Section 6
  at them.

## 0. Imports & paths

In [ ]:
import json
import math
import os
from collections import OrderedDict
from pathlib import Path

import py3Dmol
from IPython.display import HTML, display

# Not guarded on purpose: the engine cells in section 7 import pandas unconditionally, so
# `pd = None` was never a usable fallback. Swallowing the error here leaves a
# half-built pandas in sys.modules, and the real failure resurfaces later as
# "partially initialized module 'pandas' has no attribute 'core'".
import pandas as pd

# --- paths -------------------------------------------------------------------
# Everything is derived from where this notebook lives, so the whole folder can be
# moved or copied to another machine without editing anything.
# Jupyter/VSCode start the kernel in the notebook's own directory; `_dh[0]` is that
# directory even if a later cell changes the working directory. This notebook is
# expected to live in <repo>/notebooks/, so the repo root (which holds Inputs/,
# Outputs/, functions/, params/ and the settings_* folders) is one level up. If the
# notebook is run from somewhere else, fall back to its own directory.
NOTEBOOK_DIR = Path(globals().get("_dh", [Path.cwd()])[0]).resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

INPUTS = PROJECT_ROOT / "Inputs"                    # target structures
OUTPUTS = PROJECT_ROOT / "Outputs"                  # designs land here
SETTINGS_TARGET = PROJECT_ROOT / "settings_target"  # exported target JSONs
SETTINGS_ADVANCED = PROJECT_ROOT / "settings_advanced"  # engine settings, written by 5b
FILTERS = PROJECT_ROOT / "settings_filters"         # MPNN filter thresholds

for _p in (INPUTS, OUTPUTS, SETTINGS_TARGET, SETTINGS_ADVANCED, FILTERS):
    _p.mkdir(parents=True, exist_ok=True)

print(f"py3Dmol      {py3Dmol.__version__ if hasattr(py3Dmol, '__version__') else '(version n/a)'}")
print(f"PROJECT_ROOT {PROJECT_ROOT}")
for _name, _p in [("Inputs", INPUTS), ("Outputs", OUTPUTS),
                  ("settings_target", SETTINGS_TARGET),
                  ("settings_advanced", SETTINGS_ADVANCED), ("settings_filters", FILTERS)]:
    print(f"  {_name:16s} {len(list(_p.iterdir()))} entr{'y' if len(list(_p.iterdir())) == 1 else 'ies'}")

## Helper functions
Collapse it (gutter arrow); nothing below depends on reading it.

In [ ]:
# =============================================================================
# Helper functions — nothing here needs editing to run a design.
# Collapse this cell (click the gutter arrow) and work from Section 1 down.
#
#   PDB parsing & validation ... read_pdb_ca, parse_hotspots, validate_templates, summarize
#   3D viewer ................. show_templates, show_template, save_templates_html
#   Diagnostics ............... viewer_selftest
#   Export .................... build_target_json, write_target_json
#   Runtime setup ............. check_runtime, init_pyrosetta
# =============================================================================

# --- PDB parsing & validation ------------------------------------------------

VALID_STRATEGIES = ("target", "detarget")


def read_pdb_ca(pdb_path):
    '''Parse first model of a PDB. Returns (chain_order, residues, atom_lines).

    residues: OrderedDict[(chain, resnum)] -> dict(resname, x, y, z)
    '''
    chain_order, residues, atom_lines = [], OrderedDict(), []
    with open(pdb_path) as fh:
        for line in fh:
            rec = line[:6]
            if rec == "ENDMDL":
                break
            if rec != "ATOM  ":
                continue
            atom_lines.append(line.rstrip("\n"))
            if line[12:16].strip() != "CA":
                continue
            chain = line[21]
            try:
                resnum = int(line[22:26])
            except ValueError:
                continue
            if chain not in chain_order:
                chain_order.append(chain)
            residues[(chain, resnum)] = {
                "resname": line[17:20].strip(),
                "xyz": (float(line[30:38]), float(line[38:46]), float(line[46:54])),
            }
    return chain_order, residues, atom_lines


def parse_hotspots(spec, default_chain):
    '''Expand a ColabDesign hotspot string into [(chain, resnum), ...].'''
    out = []
    spec = (spec or "").strip()
    if not spec:
        return out
    for token in spec.split(","):
        token = token.strip()
        if not token:
            continue
        lo, hi = token.split("-") if "-" in token else (token, None)
        if lo[0].isalpha():
            chain, start = lo[0], int(lo[1:])
        else:
            chain, start = default_chain, int(lo)
        if hi is None:
            end = start
        else:
            end = int(hi[1:] if hi[0].isalpha() else hi)
        out.extend((chain, r) for r in range(start, end + 1))
    return out


def validate_templates(templates, general, verbose=True):
    '''Normalize + validate. Returns (validated_templates, errors, warnings).'''
    target_chains = [c.strip() for c in general["chains"].split(",") if c.strip()]
    default_chain = target_chains[0]
    errors, warnings, validated = [], [], []
    seen_names = set()

    for i, raw in enumerate(templates, start=1):
        tpl = dict(raw)
        tag = tpl.get("template_name") or f"<template #{i}>"

        if not tpl.get("template_name"):
            errors.append(f"[{tag}] template_name is empty")
        elif tpl["template_name"] in seen_names:
            errors.append(f"[{tag}] duplicate template_name — outputs would overwrite each other")
        else:
            seen_names.add(tpl["template_name"])

        # strategy: '' -> 'target' (same default the pipeline applies)
        strategy = (tpl.get("template_targeting_strategy") or "").strip() or "target"
        if strategy not in VALID_STRATEGIES:
            errors.append(f"[{tag}] template_targeting_strategy={strategy!r} not in {VALID_STRATEGIES}")
        tpl["template_targeting_strategy"] = strategy

        # loss_weighting must survive int()
        try:
            int(str(tpl.get("loss_weighting", "1")))
        except (TypeError, ValueError):
            errors.append(
                f"[{tag}] loss_weighting={tpl.get('loss_weighting')!r} is not integer-valued; "
                "step_multi casts it with int() and would raise"
            )

        # filters file
        fpath = tpl.get("template_mpnn_filters_path", "")
        if not fpath or not Path(fpath).is_file():
            errors.append(f"[{tag}] template_mpnn_filters_path not found: {fpath}")

        # structure
        pdb_path = Path(tpl.get("template_pdb", ""))
        if not pdb_path.is_file():
            errors.append(f"[{tag}] template_pdb not found: {pdb_path}")
            validated.append(tpl)
            continue

        chain_order, residues, _ = read_pdb_ca(pdb_path)
        tpl["_chains"] = chain_order
        tpl["_residues"] = residues

        missing_chains = [c for c in target_chains if c not in chain_order]
        if missing_chains:
            errors.append(
                f"[{tag}] chain(s) {missing_chains} from GENERAL['chains'] absent; PDB has {chain_order}"
            )

        target_res = [k for k in residues if k[0] in target_chains]
        tpl["_target_length"] = len(target_res)
        if target_res:
            nums = [r for _, r in target_res]
            tpl["_resnum_range"] = (min(nums), max(nums))
        else:
            tpl["_resnum_range"] = (None, None)

        extra = [c for c in chain_order if c not in target_chains]
        if extra:
            warnings.append(
                f"[{tag}] chain(s) {extra} present in the PDB but not in GENERAL['chains'] — "
                "they are ignored during design"
            )

        # hotspots
        try:
            hs = parse_hotspots(tpl.get("template_hostspot_residues", ""), default_chain)
        except (ValueError, IndexError) as exc:
            errors.append(f"[{tag}] could not parse template_hostspot_residues: {exc}")
            hs = []
        tpl["_hotspots"] = hs

        unresolved = [f"{c}{r}" for c, r in hs if (c, r) not in residues]
        if unresolved:
            errors.append(
                f"[{tag}] hotspot residue(s) absent from the PDB: {', '.join(unresolved)} "
                "(prep_inputs would raise AssertionError)"
            )
        if not hs:
            warnings.append(f"[{tag}] no hotspots set — ColabDesign will choose the interface freely")

        validated.append(tpl)

    if verbose:
        for w in warnings:
            print(f"WARNING  {w}")
        for e in errors:
            print(f"ERROR    {e}")
        if not errors:
            print(f"OK       {len(validated)} template(s) validated, no blocking problems")
    return validated, errors, warnings


def summarize(templates):
    rows = []
    for t in templates:
        lo, hi = t.get("_resnum_range", (None, None))
        rows.append({
            "name": t.get("template_name"),
            "strategy": t.get("template_targeting_strategy"),
            "weight": t.get("loss_weighting"),
            "chains": ",".join(t.get("_chains", [])),
            "n_res": t.get("_target_length"),
            "resnum_range": f"{lo}-{hi}" if lo is not None else "-",
            "n_hotspots": len(t.get("_hotspots", [])),
            "pdb": Path(t.get("template_pdb", "")).name,
        })
    if pd is not None:
        return pd.DataFrame(rows)
    for r in rows:
        print(r)
    return rows


# --- 3D viewer ---------------------------------------------------------------

# Distinct cartoon colour per template; hotspots are always red.
PALETTE = ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3", "#937860"]
HOTSPOT_COLOR = "#E24A33"
OFF_TARGET_COLOR = "#D9D9D9"


def _hotspots_by_chain(tpl):
    by_chain = {}
    for ch, rn in (tpl.get("_hotspots") or []):
        by_chain.setdefault(ch, []).append(rn)
    return by_chain


def _draw_panel(view, tpl, target_chains, color, cell=None,
                show_surface=False, stick_radius=0.35, show_resi_labels=False):
    """Draw one template into `view`.

    `cell` is the (row, col) of a viewergrid, or None for a single-panel view.
    py3Dmol applies a call to *every* panel unless `viewer=` names one, so the kwarg
    must be omitted entirely rather than passed as None — and it must be a keyword:
    passed positionally it leaks into the 3Dmol call as a stray argument.
    """
    kw = {} if cell is None else {"viewer": cell}

    view.addModel(Path(tpl["template_pdb"]).read_text(), "pdb", **kw)

    # base: everything pale, target chain(s) in the template colour
    view.setStyle({}, {"cartoon": {"color": OFF_TARGET_COLOR, "opacity": 0.55}}, **kw)
    view.setStyle({"chain": target_chains},
                  {"cartoon": {"color": color, "opacity": 0.85}}, **kw)

    # hotspots: side-chain sticks that visibly protrude from the cartoon
    for ch, nums in _hotspots_by_chain(tpl).items():
        sel = {"chain": ch, "resi": nums}
        view.addStyle(sel, {"cartoon": {"color": HOTSPOT_COLOR}}, **kw)
        view.addStyle(sel, {"stick": {"radius": stick_radius,
                                      "colorscheme": "redCarbon"}}, **kw)
        view.addStyle({"chain": ch, "resi": nums, "atom": "CA"},
                      {"sphere": {"radius": stick_radius * 1.7,
                                  "color": HOTSPOT_COLOR}}, **kw)
        if show_surface:
            view.addSurface("VDW", {"opacity": 0.30, "color": HOTSPOT_COLOR}, sel, **kw)
        if show_resi_labels:
            view.addResLabels(sel, {"fontSize": 10, "fontColor": "black",
                                    "backgroundColor": "white",
                                    "backgroundOpacity": 0.7,
                                    "showBackground": True}, **kw)

    view.zoomTo({"chain": target_chains}, **kw)


def _header_html(tpl, color, width):
    """Script-free header: PDB filename on top, template details underneath."""
    pdb_name = Path(tpl.get("template_pdb", "")).name or "(no pdb)"
    n_hs = len(tpl.get("_hotspots") or [])
    lo, hi = tpl.get("_resnum_range", (None, None))
    bits = [
        f"<b>{tpl.get('template_name', '?')}</b>",
        f"chain {','.join(tpl.get('_chains', [])) or '?'}",
        f"{tpl.get('_target_length', '?')} res",
        (f"numbering {lo}–{hi}" if lo is not None else "-"),
        (f"<span style='color:{HOTSPOT_COLOR}'><b>{n_hs} hotspot"
         f"{'s' if n_hs != 1 else ''}</b></span>" if n_hs else
         "<span style='color:#999'>no hotspots</span>"),
        tpl.get("template_targeting_strategy", "target"),
    ]
    return (
        f"<div style='flex:0 0 {width}px;box-sizing:border-box;padding:8px 10px;"
        "border:1px solid #ddd;border-bottom:none;border-radius:6px 6px 0 0;"
        "background:#fafafa;font-family:system-ui,sans-serif;'>"
        f"<div style='font-size:15px;font-weight:600;color:#222;"
        f"border-left:5px solid {color};padding-left:8px;line-height:1.3;'>"
        f"{pdb_name}</div>"
        "<div style='font-size:11px;color:#555;margin-top:3px;padding-left:13px;'>"
        + " &nbsp;·&nbsp; ".join(bits) +
        "</div></div>"
    )


def _header_row(templates, panel_w, ncols):
    """One flex row of headers, each exactly one panel wide so it sits over its panel.

    ponytail: only the first grid row lines up when templates wrap onto several rows —
    a single canvas cannot have HTML interleaved between its rows. Irrelevant at the
    2-3 templates this is built for; for more than that, drop the headers and
    read the in-canvas labels instead.
    """
    chips = "".join(_header_html(t, PALETTE[i % len(PALETTE)], panel_w)
                    for i, t in enumerate(templates))
    return (f"<div style='display:flex;flex-wrap:wrap;margin-top:10px;"
            f"max-width:{panel_w * ncols}px;'>{chips}</div>")


def show_templates(templates, general=None, panel=(420, 380), ncols=None, linked=False,
                   show_surface=False, stick_radius=0.35, show_resi_labels=False):
    """All templates side by side, as one viewergrid in a single output.

    One viewer object means one <script>, which is the render path VSCode handles most
    reliably. The headers go out as a separate HTML output above it — wrapping py3Dmol's
    own HTML inside a document of ours stops its bootstrap script from running.

    panel  -- (width, height) of each panel in px
    ncols  -- columns; defaults to one row holding every template
    linked -- True ties the cameras together, so rotating one panel rotates all of them.
              Off by default: linked panels share a zoom, which crops templates that are
              not already in a common frame. Turn it on for orthologs of equal length.
    """
    general = general or GENERAL
    templates = [t for t in templates if Path(t.get("template_pdb", "")).is_file()]
    if not templates:
        print("No readable template PDBs to display.")
        return

    if not any(t.get("_hotspots") for t in templates):
        print("Note: no hotspots set on any template — the red stick styling "
              "appears once template_hostspot_residues is filled in (Section 2).")

    target_chains = [c.strip() for c in general["chains"].split(",") if c.strip()]
    n = len(templates)
    ncols = max(1, min(ncols or n, n))
    nrows = math.ceil(n / ncols)

    view = py3Dmol.view(viewergrid=(nrows, ncols), linked=linked,
                        width=panel[0] * ncols, height=panel[1] * nrows)
    for i, tpl in enumerate(templates):
        _draw_panel(view, tpl, target_chains, PALETTE[i % len(PALETTE)],
                    cell=(i // ncols, i % ncols), show_surface=show_surface,
                    stick_radius=stick_radius, show_resi_labels=show_resi_labels)
        # in-canvas name, so an exported PNG or a wrapped header row stays readable
        view.addLabel(tpl.get("template_name", "?"),
                      {"useScreen": True, "screenOffset": {"x": 6, "y": 6},
                       "fontSize": 12, "fontColor": "black", "backgroundColor": "white",
                       "backgroundOpacity": 0.85, "inFront": True},
                      {}, viewer=(i // ncols, i % ncols))
    view.setBackgroundColor("white")

    display(HTML(_header_row(templates, panel[0], ncols)))
    view.show()
    return view


def show_template(tpl, general=None, size=(760, 560), show_surface=False,
                  spin=False, stick_radius=0.38, show_resi_labels=True):
    """Single template, larger, residue labels on by default."""
    general = general or GENERAL
    target_chains = [c.strip() for c in general["chains"].split(",") if c.strip()]
    view = py3Dmol.view(width=size[0], height=size[1])
    _draw_panel(view, tpl, target_chains, PALETTE[0], cell=None,
                show_surface=show_surface, stick_radius=stick_radius,
                show_resi_labels=show_resi_labels)
    view.setBackgroundColor("white")
    if spin:
        view.spin(True)
    display(HTML(_header_row([tpl], size[0], 1)))
    view.show()
    return view


def save_templates_html(templates, path, general=None, panel=(420, 380), ncols=None,
                        linked=False, show_surface=False, stick_radius=0.35,
                        show_resi_labels=False):
    """Write the same grid to a standalone HTML file (3Dmol.js from CDN).

    Scripts run normally in a real browser page, so this always works even when the
    notebook renderer does not.
    """
    general = general or GENERAL
    templates = [t for t in templates if Path(t.get("template_pdb", "")).is_file()]
    if not templates:
        print("No readable template PDBs to write.")
        return

    target_chains = [c.strip() for c in general["chains"].split(",") if c.strip()]
    n = len(templates)
    ncols = max(1, min(ncols or n, n))
    nrows = math.ceil(n / ncols)

    view = py3Dmol.view(viewergrid=(nrows, ncols), linked=linked,
                        width=panel[0] * ncols, height=panel[1] * nrows)
    for i, tpl in enumerate(templates):
        _draw_panel(view, tpl, target_chains, PALETTE[i % len(PALETTE)],
                    cell=(i // ncols, i % ncols), show_surface=show_surface,
                    stick_radius=stick_radius, show_resi_labels=show_resi_labels)
        view.addLabel(tpl.get("template_name", "?"),
                      {"useScreen": True, "screenOffset": {"x": 6, "y": 6},
                       "fontSize": 12, "fontColor": "black", "backgroundColor": "white",
                       "backgroundOpacity": 0.85, "inFront": True},
                      {}, viewer=(i // ncols, i % ncols))
    view.setBackgroundColor("white")

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("<!doctype html><html><head><meta charset='utf-8'>"
                    "<title>BindCraft templates</title></head>"
                    "<body style='margin:16px;background:#fff;'>"
                    + _header_row(templates, panel[0], ncols)
                    + view._make_html() + "</body></html>")
    print(f"wrote {path}")
    return path


# --- Diagnostics -------------------------------------------------------------

def viewer_selftest():
    """Minimal py3Dmol render - isolates 'my code is wrong' from 'the webview can't
    run 3Dmol at all'.

    Emits the simplest possible viewer: 3 atoms, one style call, nothing else. If this
    panel is BLANK, the problem is environmental (notebook not trusted, webview CSP
    blocking the CDN, kernel/extension state) and no change to the viewer code will fix
    it. If a small molecule appears, 3Dmol works and the fault is in the styling code.
    """
    tiny = (
        "ATOM      1  N   GLY A   1       0.000   0.000   0.000  1.00  0.00           N\n"
        "ATOM      2  CA  GLY A   1       1.458   0.000   0.000  1.00  0.00           C\n"
        "ATOM      3  C   GLY A   1       2.009   1.420   0.000  1.00  0.00           C\n"
        "END\n"
    )
    v = py3Dmol.view(width=320, height=240)
    v.addModel(tiny, "pdb")
    v.setStyle({}, {"sphere": {"radius": 0.5}})
    v.zoomTo()
    print("Three spheres below means 3Dmol.js is working in this notebook.")
    print("If the area below is blank/empty, it is an environment problem - see docstring.")
    v.show()


# --- Export ------------------------------------------------------------------

def build_target_json(general, templates):
    payload = {"general_information": dict(general)}
    for i, tpl in enumerate(templates, start=1):
        payload[f"template_{i}_information"] = {
            "template_name":               tpl["template_name"],
            "template_pdb":                tpl["template_pdb"],
            "template_hostspot_residues":  tpl.get("template_hostspot_residues", ""),
            "template_targeting_strategy": tpl.get("template_targeting_strategy", "target"),
            "template_mpnn_filters_path":  tpl["template_mpnn_filters_path"],
            "loss_weighting":              str(tpl.get("loss_weighting", "1")),
        }
    return payload


def write_target_json(general, templates, filename=None, overwrite=False):
    payload = build_target_json(general, templates)
    filename = filename or f"{general['binder_name']}.json"
    out = SETTINGS_TARGET / filename
    if out.exists() and not overwrite:
        raise FileExistsError(f"{out} exists — pass overwrite=True to replace it")
    Path(general["design_path"]).mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(payload, indent=4))
    print(f"wrote {out}")
    return out


# --- Runtime setup -----------------------------------------------------------
# Everything below needs a GPU node. JAX / ColabDesign / PyRosetta are imported
# lazily *inside* these functions so Sections 0-4 stay runnable on a login node.

AF_MODEL_NAMES = {True:  [f"model_{k}_multimer_v3" for k in range(1, 6)],
                  False: [f"model_{k}_ptm" for k in range(1, 6)]}


GPU_HELP = """No GPU available to JAX — binder design needs one.

  * On a cluster: request a GPU node (`srun --gres=gpu:1 --pty bash`) and restart the
    kernel there. `nvidia-smi` showing a card is not enough — the kernel must be on the
    node that holds the allocation.
  * Seen on this cluster: cuInit intermittently fails with CUDA_ERROR_NOT_INITIALIZED
    for several minutes at a time while `nvidia-smi` stays healthy, then recovers on its
    own. If the setup is unchanged, wait and re-run this cell before changing
    anything. Switching conda environments does not help — the fault is below Python."""


def check_gpu(verbose=True):
    """The pipeline's check_jax_gpu(), raising something actionable instead of exit().

    jax.devices() itself raises when no backend initialises at all, which is the most
    common failure — so that case has to be caught here or the advice below never prints.
    """
    import jax
    try:
        devices = jax.devices()
    except RuntimeError as exc:
        raise RuntimeError(f"{GPU_HELP}\n\n  JAX reported: {exc}") from exc
    if verbose:
        for d in devices:
            print(f"    {d.device_kind} ({d.platform})")
    if not any(d.platform == "gpu" for d in devices):
        raise RuntimeError(f"{GPU_HELP}\n\n  JAX sees only: "
                           + ", ".join(f"{d.device_kind} ({d.platform})" for d in devices))
    return devices


def check_af_params(af_params_dir, use_multimer=True):
    """ColabDesign loads <af_params_dir>/params/params_<model>.npz — check they are there.

    This is the one dependency too large to keep next to the notebook (~5 GB). Point
    AF_PARAMS_DIR at any existing AlphaFold2 / BindCraft / ColabDesign weights directory.
    """
    root = Path(af_params_dir)
    names = AF_MODEL_NAMES[bool(use_multimer)]
    missing = [n for n in names if not (root / "params" / f"params_{n}.npz").is_file()]
    if missing:
        raise FileNotFoundError(
            f"AlphaFold2 weights missing under {root / 'params'}: "
            + ", ".join(f"params_{n}.npz" for n in missing)
            + "\nSet AF_PARAMS_DIR to a directory that contains a 'params/' folder with these, "
              "or fetch them:\n"
              "  mkdir -p params && cd params && "
              "wget -qO- https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar | tar x"
        )
    return [str(root / "params" / f"params_{n}.npz") for n in names]


def check_executables(**paths):
    """Each path must exist and be executable — same check the pipeline makes, named."""
    for label, path in paths.items():
        if not Path(path).is_file():
            raise FileNotFoundError(f"{label} not found: {path}")
        if not os.access(path, os.X_OK):
            raise PermissionError(f"{label} is not executable: {path}\n  fix: chmod +x {path}")
    return True


def check_runtime(af_params_dir, advanced_json, dssp_path, dalphaball_path,
                  use_multimer=None, verbose=True):
    """Run every startup check `multi_binder_design.main()` makes, before committing a GPU.

    Returns (advanced_settings, design_models, prediction_models, multimer_validation).
    Raises on anything that would otherwise fail deep inside a trajectory.
    """
    say = print if verbose else (lambda *a, **k: None)

    say("GPU")
    devices = check_gpu(verbose=verbose)

    say("advanced settings")
    with open(advanced_json) as fh:
        advanced_settings = json.load(fh)
    say(f"    {Path(advanced_json).resolve()}  ({len(advanced_settings)} keys)")

    if use_multimer is None:
        use_multimer = advanced_settings["use_multimer_design"]

    say("AlphaFold2 weights")
    found = check_af_params(af_params_dir, use_multimer)
    say(f"    {len(found)} x {'multimer_v3' if use_multimer else 'ptm'} under {Path(af_params_dir) / 'params'}")

    say("executables")
    check_executables(dssp=dssp_path, DAlphaBall=dalphaball_path)
    say(f"    dssp        {dssp_path}")
    say(f"    DAlphaBall  {dalphaball_path}")

    # the pipeline overwrites these three from its own folder layout; set them explicitly
    advanced_settings["af_params_dir"] = str(af_params_dir)
    advanced_settings["dssp_path"] = str(dssp_path)
    advanced_settings["dalphaball_path"] = str(dalphaball_path)

    # normalize omit_AAs exactly as perform_advanced_settings_check() does
    if advanced_settings.get("omit_AAs") in (None, False, ""):
        advanced_settings["omit_AAs"] = None
    elif isinstance(advanced_settings["omit_AAs"], str):
        advanced_settings["omit_AAs"] = advanced_settings["omit_AAs"].strip()

    # load_af2_models(): design and prediction must not share models
    if use_multimer:
        design_models, prediction_models, multimer_validation = [0, 1, 2, 3, 4], [0, 1], False
    else:
        design_models, prediction_models, multimer_validation = [0, 1], [0, 1, 2, 3, 4], True

    say("model split")
    say(f"    design {design_models}  prediction {prediction_models}  "
        f"multimer_validation={multimer_validation}")
    say(f"\nready — {len(devices)} device(s), omit_AAs={advanced_settings['omit_AAs']!r}")
    return advanced_settings, design_models, prediction_models, multimer_validation


def init_pyrosetta(advanced_settings, extra_flags=""):
    """Initialise PyRosetta with the pipeline's flags. Safe to re-run; it re-inits."""
    import pyrosetta as pr
    flags = (
        "-ignore_unrecognized_res -ignore_zero_occupancy -mute all "
        f"-holes:dalphaball {advanced_settings['dalphaball_path']} "
        "-corrections::beta_nov16 true -relax:default_repeats 1 "
        + extra_flags
    ).strip()
    pr.init(flags)
    print(f"PyRosetta initialised\n  {flags}")
    return pr

## 1. General settings

Binder-level settings, shared by every template. These map 1:1 onto the
`general_information` block of a BindCraft target JSON.

| key | meaning |
|---|---|
| `design_path` | output directory; trajectory/MPNN/final PDBs and all CSVs land here |
| `binder_name` | prefix for every design name (`<name>_l<length>_s<seed>`) |
| `chains` | **target** chain(s) to bind, comma-separated. Must exist in *every* template PDB |
| `binder_chain` | chain ID assigned to the designed binder; ColabDesign always appends it as `B` |
| `lengths` | `[min, max]`; each trajectory samples a length uniformly from this range |
| `number_of_final_designs` | stop once this many designs pass all filters for all templates |


In [ ]:
GENERAL = {
    "design_path": str(OUTPUTS / "PDL1_duo"),
    "binder_name": "PDL1_duo",
    "chains": "A",
    "binder_chain": "B",
    "lengths": [120, 150],
    "number_of_final_designs": 100,
}

GENERAL

## 2. Template definitions

One entry per target structure. All templates share a **single binder sequence**; they
differ in the structure the binder is folded against and in how their loss is weighted.

| key | meaning |
|---|---|
| `template_name` | short label; suffixes every output PDB and names the per-template CSVs. **Must be unique** |
| `template_pdb` | path to the target structure |
| `template_hostspot_residues` | residues the binder should contact, e.g. `"47,96,118"` or `"85-92,227"`. Empty string ⇒ no restriction (ColabDesign picks the interface itself). *Note the `hostspot` typo: it is the real key name in the pipeline* |
| `template_targeting_strategy` | `"target"` (bind it) or `"detarget"` (avoid it). Empty string defaults to `"target"` |
| `template_mpnn_filters_path` | filter JSON applied to MPNN designs for this template |
| `loss_weighting` | relative weight of this template's gradient |

### Hotspot syntax

Parsed by ColabDesign's `prep_pos`: comma-separated residues and `start-end` ranges,
using **PDB author numbering** (not 0-based index). A residue may carry a chain prefix
(`"A341"`); without one it is assigned to the **first chain** in `chains`. A residue that
does not exist in the PDB raises an `AssertionError` deep inside `prep_inputs`, which is
exactly what Section 3 checks for up front.

### On `loss_weighting`

The relative weight of this template's gradient in the combined update. It is read with
`float(...)` inside a `try`, so fractional strings such as `"0.5"` are fine and anything
unparseable silently falls back to `1.0`. Keep it at `"1"` for symmetric ortholog pairs.

Whether the weight is the only thing balancing the templates depends on
`multi_grad_rms_norm` (section 7b). With it **on**, each template's gradient is first
divided by its own RMS, so `loss_weighting` alone decides the balance. With it **off**,
the gradients are combined raw, and a structurally dissimilar template with a larger
gradient magnitude can dominate regardless of its weight.

In [ ]:
TEMPLATES = [
    {
        "template_name":              "hPDL1",
        "template_pdb":               str(INPUTS / "PDL1" / "hPDL1.pdb"),
        "template_hostspot_residues": "",
        "template_targeting_strategy": "target",
        "template_mpnn_filters_path": str(FILTERS / "default_filters.json"),
        "loss_weighting":             "1",
    },
    {
        "template_name":              "mPDL1",
        "template_pdb":               str(INPUTS / "PDL1" / "mPDL1.pdb"),
        "template_hostspot_residues": "",
        "template_targeting_strategy": "target",
        "template_mpnn_filters_path": str(FILTERS / "default_filters.json"),
        "loss_weighting":             "1",
    },
]

print(f"{len(TEMPLATES)} templates defined: " + ", ".join(t["template_name"] for t in TEMPLATES))

## 3. Parsing & validation

A minimal PDB reader that mirrors what ColabDesign actually sees: **first model only**,
`ATOM` records only, one entry per residue keyed on its CA atom. Validation then checks
every assumption that would otherwise fail at runtime: missing files, missing chains,
hotspots that do not resolve, duplicate names, unparseable weights.

In [ ]:
TEMPLATES_V, ERRORS, WARNINGS = validate_templates(TEMPLATES, GENERAL)
summarize(TEMPLATES_V)

## 4. Template viewer

All templates are drawn **side by side** in a single 3D panel grid, with an HTML header
over each one naming its PDB file (plus template name, chain, residue count, numbering
range, hotspot count and strategy).

Hotspot residues are drawn as **full side-chain sticks with red carbons**, protruding out
of the cartoon, plus a sphere on each CA. The VDW surface is **off by default** because it
buries the sticks, which defeats the point; pass `show_surface=True` to judge the
shape of a patch rather than the identity of the residues.

> **Blank panel?** Run `viewer_selftest()`, which draws 3 spheres and nothing else. If those
> are blank too the problem is environmental (notebook not trusted, the VSCode webview
> blocking the 3Dmol.js CDN, kernel needing a reload) and no change to the viewer code
> will help. `save_templates_html(TEMPLATES_V, "templates.html")` always works: it writes
> the same grid as a plain web page, outside the notebook renderer entirely.

In [ ]:
# show_surface=True adds a translucent VDW patch over the hotspots
show_templates(TEMPLATES_V);
# single templates can be viewed with:
# show_template(TEMPLATES_V[0]);

## 5. The target definition

`build_target_json()` produces the `general_information` + `template_N_information` dict
that `collect_target_information()` consumes. **In this notebook the dict is all that is
needed**; nothing reads the file back. Writing it is still worth doing as a record of
what was run, and it is what `--settings` takes if the same job is later submitted with
`sbatch`.

In [ ]:
TARGET = build_target_json(GENERAL, TEMPLATES_V)
print(json.dumps(TARGET, indent=4))

In [ ]:
# Optional: keep a copy on disk as the record of this run.
if ERRORS:
    print(f"{len(ERRORS)} validation error(s) outstanding; not writing the JSON.")
    TARGET_JSON = None
else:
    # the path is recorded in every trajectory_csv row, so keep it
    TARGET_JSON = write_target_json(GENERAL, TEMPLATES_V, overwrite=True)

## 5b. Advanced settings

The engine settings live here rather than in a hand-maintained JSON. This cell
writes them to `settings_advanced/` and section 6 reads that file back, so the
notebook is the source of truth and the file is a generated artifact.

`af_params_dir`, `dssp_path` and `dalphaball_path` are left empty on purpose:
`check_runtime()` overwrites all three with paths derived from this folder.

In [ ]:
# @title Advanced settings, defined here rather than in a settings file
# The full engine settings dict. Written to disk below and read back by
# check_runtime() in section 6, so the notebook is the source of truth and the
# file on disk is a generated artifact. All 96 keys the engine reads are present.

ADVANCED = {
    # --- sequence, model and template handling ---
    "omit_AAs":                               "C",
    "force_reject_AA":                        False,
    "use_multimer_design":                    True,
    "design_algorithm":                       "4stage_duo",
    "sample_models":                          True,
    "rm_template_seq_design":                 False,
    "rm_template_seq_predict":                False,
    "rm_template_sc_design":                  False,
    "rm_template_sc_predict":                 False,
    "antidesign":                             False,
    "predict_initial_guess":                  True,
    "predict_bigbang":                        False,

    # --- trajectory: stage lengths and detargeting decay ---
    "buffer_iterations":                      0,
    "use_loss_decay":                         False,
    "default_detargeting_decay":              0.25,
    "decay_iters":                            0,
    "test_iterations":                        50,
    "soft_iterations":                        75,
    "temporary_iterations":                   45,
    "hard_iterations":                        10,
    "greedy_iterations":                      0,
    "greedy_percentage":                      5,
    "mcmc_iterations":                        200,
    "save_design_animations":                 False,
    "save_design_trajectory_plots":           True,

    # --- core AF2 loss weights ---
    "weights_plddt":                          0.1,
    "weights_pae_intra":                      0.4,
    "weights_pae_inter":                      0.1,
    "weights_con_intra":                      1.0,
    "weights_con_inter":                      1.5,

    # --- optional target losses, and the per-term detargeting scales ---
    "use_target_plddt_loss":                  True,
    "weights_target_plddt":                   0.1,
    "detargeting_scaling_weights_pae_intra":  1,
    "detargeting_scaling_weights_plddt":      1,
    "detargeting_scaling_weights_pae_inter":  0,
    "detargeting_scaling_weights_con_intra":  1,
    "detargeting_scaling_weights_con_inter":  0,
    "use_i_ptm_loss":                         True,
    "weights_iptm":                           0.1,
    "detargeting_scaling_weights_iptm":       -1,

    # --- contact definitions ---
    "intra_contact_distance":                 14.0,
    "inter_contact_distance":                 20.0,
    "intra_contact_number":                   2,
    "inter_contact_number":                   2,

    # --- helicity, radius of gyration, etc. ---
    "weights_helicity":                       -0.5,
    "random_helicity":                        False,
    "use_rg_loss":                            True,
    "weights_rg":                             0.3,

    # --- ProteinMPNN ---
    "enable_mpnn":                            True,
    "mpnn_fix_interface":                     True,
    "num_seqs":                               20,
    "max_mpnn_sequences":                     2,
    "sampling_temp":                          0.1,
    "sample_seq_parallel":                    20,
    "backbone_noise":                         0.0,
    "model_path":                             "v_48_020",
    "mpnn_weights":                           "soluble",
    "save_mpnn_fasta":                        False,

    # --- AF2 recycles and beta-sheet handling ---
    "num_recycles_design":                    1,
    "num_recycles_validation":                3,
    "optimise_beta":                          True,
    "optimise_beta_extra_soft":               0,
    "optimise_beta_extra_temp":               0,
    "optimise_beta_recycles_design":          3,
    "optimise_beta_recycles_valid":           3,

    # --- output cleanup and run monitoring ---
    "remove_unrelaxed_trajectory":            True,
    "remove_unrelaxed_complex":               True,
    "remove_binder_monomer":                  True,
    "zip_animations":                         True,
    "zip_plots":                              True,
    "save_trajectory_pickle":                 False,
    "max_trajectories":                       False,
    "enable_rejection_check":                 True,
    "acceptance_rate":                        0.01,
    "start_monitoring":                       300,

    # --- executables and weights: check_runtime() overwrites all three in section 6 ---
    "af_params_dir":                          "params",
    "dssp_path":                              "functions/dssp",
    "dalphaball_path":                        "functions/DAlphaBall.gcc",

    # --- multi-template aggregation (section 7b can override these at runtime) ---
    "multi_shuffle_templates":                False,
    "multi_grad_rms_norm":                    False,
    "multi_grad_rms_eps":                     1e-08,
    "multi_dynamic_weight_beta":              3.0,
    "multi_bottleneck_alpha":                 12.0,
    "multi_stage1_warmup_iters":              10,
}

# --- write it out ------------------------------------------------------------
ADVANCED_NAME = "duo_notebook_generated"
ADVANCED_JSON = SETTINGS_ADVANCED / f"{ADVANCED_NAME}.json"

with open(ADVANCED_JSON, "w") as _fh:
    json.dump(ADVANCED, _fh, indent=4)

print(f"new advanced setting written at {ADVANCED_JSON}")

## 6. Runtime setup

**Sections 0–5 need only a CPU. From here on a GPU is required.**

`check_runtime()` makes every startup check `multi_binder_design.main()` makes
(GPU, AF2 weights, dssp/DAlphaBall execute permissions) and returns an
`advanced_settings`.

In [ ]:
# @title Runtime checks
AF_PARAMS_DIR = PROJECT_ROOT                              # parent of params/
# ADVANCED_JSON comes from section 5b
DSSP          = PROJECT_ROOT / "functions" / "dssp"
DALPHABALL    = PROJECT_ROOT / "functions" / "DAlphaBall.gcc"

advanced_settings, design_models, prediction_models, multimer_validation = check_runtime(
    af_params_dir=AF_PARAMS_DIR,
    advanced_json=ADVANCED_JSON,
    dssp_path=DSSP,
    dalphaball_path=DALPHABALL,
)

# Everything is set, BindCraft is ready to run!

## 7. The engine

The `functions/` package inlined, one cell per module, so the notebook runs with
no local imports. Collapse them; nothing below depends on reading them.

In [ ]:
#============================================================================
# Engine imports
#============================================================================
# The union of what functions/{biopython,generic,pyrosetta,multi_colabdesign}_utils.py
# and functions/__init__.py imported, deduplicated. The four cells below hold those
# modules inlined, so nothing under functions/ is imported any more and the notebook
# is self-contained. Only the conda environment comes from outside.

# --- stdlib ---
import argparse
import copy
import json
import math
import os
import pickle
import random
import re
import shutil
import time
import warnings
import zipfile
from collections import defaultdict

# --- scientific stack ---
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.special import softmax

# --- jax ---
import jax
import jax.numpy as jnp
from jax import debug

# --- colabdesign ---
from colabdesign import mk_afdesign_model, clear_mem
from colabdesign.mpnn import mk_mpnn_model
from colabdesign.af.alphafold.common import residue_constants
from colabdesign.af.loss import get_ptm, mask_loss, get_dgram_bins, _get_con_loss
from colabdesign.shared.utils import copy_dict

# --- biopython ---
from Bio import BiopythonWarning
from Bio.PDB import PDBParser, DSSP, Selection, Polypeptide, PDBIO, Select, Chain, Superimposer
from Bio.PDB.Polypeptide import is_aa
from Bio.PDB.Selection import unfold_entities
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# --- pyrosetta (initialised in section 8, not here) ---
import pyrosetta as pr
from pyrosetta.rosetta.core.io import pose_from_pose
from pyrosetta.rosetta.core.kinematics import MoveMap
from pyrosetta.rosetta.core.select import get_residues_from_subset
from pyrosetta.rosetta.core.select.residue_selector import ChainSelector
from pyrosetta.rosetta.core.simple_metrics.metrics import RMSDMetric
from pyrosetta.rosetta.protocols.analysis import InterfaceAnalyzerMover
from pyrosetta.rosetta.protocols.relax import FastRelax
from pyrosetta.rosetta.protocols.rosetta_scripts import XmlObjects
from pyrosetta.rosetta.protocols.simple_moves import AlignChainMover

# carried over from functions/__init__.py
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=DeprecationWarning)
warnings.simplefilter(action='ignore', category=BiopythonWarning)

print("engine imports ok")

In [ ]:
#============================================================================
# BioPython Utils
#============================================================================
# Inlined verbatim from functions/biopython_utils.py; imports hoisted to the cell above.

# analyze sequence composition of design
def validate_design_sequence(sequence, num_clashes, advanced_settings):
    note_array = []

    # Check if protein contains clashes after relaxation
    if num_clashes > 0:
        note_array.append('Relaxed structure contains clashes.')

    # Check if the sequence contains disallowed amino acids
    if advanced_settings["omit_AAs"]:
        restricted_AAs = advanced_settings["omit_AAs"].split(',')
        for restricted_AA in restricted_AAs:
            if restricted_AA in sequence:
                note_array.append('Contains: '+restricted_AA+'!')

    # Analyze the protein
    analysis = ProteinAnalysis(sequence)

    # Calculate the reduced extinction coefficient per 1% solution
    extinction_coefficient_reduced = analysis.molar_extinction_coefficient()[0]
    molecular_weight = round(analysis.molecular_weight() / 1000, 2)
    extinction_coefficient_reduced_1 = round(extinction_coefficient_reduced / molecular_weight * 0.01, 2)

    # Check if the absorption is high enough
    if extinction_coefficient_reduced_1 <= 2:
        note_array.append(f'Absorption value is {extinction_coefficient_reduced_1}, consider adding tryptophane to design.')

    # Join the notes into a single string
    notes = ' '.join(note_array)

    return notes

# temporary function, calculate RMSD of input PDB and trajectory target
def target_pdb_rmsd(trajectory_pdb, starting_pdb, chain_ids_string):
    # Parse the PDB files
    parser = PDBParser(QUIET=True)
    structure_trajectory = parser.get_structure('trajectory', trajectory_pdb)
    structure_starting = parser.get_structure('starting', starting_pdb)
    
    # Extract chain A from trajectory_pdb
    chain_trajectory = structure_trajectory[0]['A']
    
    # Extract the specified chains from starting_pdb
    chain_ids = chain_ids_string.split(',')
    residues_starting = []
    for chain_id in chain_ids:
        chain_id = chain_id.strip()
        chain = structure_starting[0][chain_id]
        for residue in chain:
            if is_aa(residue, standard=True):
                residues_starting.append(residue)
    
    # Extract residues from chain A in trajectory_pdb
    residues_trajectory = [residue for residue in chain_trajectory if is_aa(residue, standard=True)]
    
    # Ensure that both structures have the same number of residues
    min_length = min(len(residues_starting), len(residues_trajectory))
    residues_starting = residues_starting[:min_length]
    residues_trajectory = residues_trajectory[:min_length]
    
    # Collect CA atoms from the two sets of residues
    atoms_starting = [residue['CA'] for residue in residues_starting if 'CA' in residue]
    atoms_trajectory = [residue['CA'] for residue in residues_trajectory if 'CA' in residue]
    
    # Calculate RMSD using structural alignment
    sup = Superimposer()
    sup.set_atoms(atoms_starting, atoms_trajectory)
    rmsd = sup.rms
    
    return round(rmsd, 2)

# detect C alpha clashes for deformed trajectories
def calculate_clash_score(pdb_file, threshold=2.4, only_ca=False):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein', pdb_file)

    atoms = []
    atom_info = []  # Detailed atom info for debugging and processing

    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    if atom.element == 'H':  # Skip hydrogen atoms
                        continue
                    if only_ca and atom.get_name() != 'CA':
                        continue
                    atoms.append(atom.coord)
                    atom_info.append((chain.id, residue.id[1], atom.get_name(), atom.coord))

    tree = cKDTree(atoms)
    pairs = tree.query_pairs(threshold)

    valid_pairs = set()
    for (i, j) in pairs:
        chain_i, res_i, name_i, coord_i = atom_info[i]
        chain_j, res_j, name_j, coord_j = atom_info[j]

        # Exclude clashes within the same residue
        if chain_i == chain_j and res_i == res_j:
            continue

        # Exclude directly sequential residues in the same chain for all atoms
        if chain_i == chain_j and abs(res_i - res_j) == 1:
            continue

        # If calculating sidechain clashes, only consider clashes between different chains
        if not only_ca and chain_i == chain_j:
            continue

        valid_pairs.add((i, j))

    return len(valid_pairs)

three_to_one_map = {
    'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F',
    'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L',
    'MET': 'M', 'ASN': 'N', 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R',
    'SER': 'S', 'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y'
}

# identify interacting residues at the binder interface
def hotspot_residues(trajectory_pdb, binder_chain="B", atom_distance_cutoff=4.0):
    # Parse the PDB file
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("complex", trajectory_pdb)

    # Get the specified chain
    binder_atoms = Selection.unfold_entities(structure[0][binder_chain], 'A')
    binder_coords = np.array([atom.coord for atom in binder_atoms])

    # Get atoms and coords for the target chain
    target_atoms = Selection.unfold_entities(structure[0]['A'], 'A')
    target_coords = np.array([atom.coord for atom in target_atoms])

    # Build KD trees for both chains
    binder_tree = cKDTree(binder_coords)
    target_tree = cKDTree(target_coords)

    # Prepare to collect interacting residues
    interacting_residues = {}

    # Query the tree for pairs of atoms within the distance cutoff
    pairs = binder_tree.query_ball_tree(target_tree, atom_distance_cutoff)

    # Process each binder atom's interactions
    for binder_idx, close_indices in enumerate(pairs):
        binder_residue = binder_atoms[binder_idx].get_parent()
        binder_resname = binder_residue.get_resname()

        # Convert three-letter code to single-letter code using the manual dictionary
        if binder_resname in three_to_one_map:
            aa_single_letter = three_to_one_map[binder_resname]
            for close_idx in close_indices:
                target_residue = target_atoms[close_idx].get_parent()
                interacting_residues[binder_residue.id[1]] = aa_single_letter

    return interacting_residues

# calculate secondary structure percentage of design
def calc_ss_percentage(pdb_file, advanced_settings, chain_id="B", atom_distance_cutoff=4.0):
    # Parse the structure
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('protein', pdb_file)
    model = structure[0]  # Consider only the first model in the structure

    # Calculate DSSP for the model
    dssp = DSSP(model, pdb_file, dssp=advanced_settings["dssp_path"])

    # Prepare to count residues
    ss_counts = defaultdict(int)
    ss_interface_counts = defaultdict(int)
    plddts_interface = []
    plddts_ss = []

    # Get chain and interacting residues once
    chain = model[chain_id]
    interacting_residues = set(hotspot_residues(pdb_file, chain_id, atom_distance_cutoff).keys())

    for residue in chain:
        residue_id = residue.id[1]
        if (chain_id, residue_id) in dssp:
            ss = dssp[(chain_id, residue_id)][2]  # Get the secondary structure
            ss_type = 'loop'
            if ss in ['H', 'G', 'I']:
                ss_type = 'helix'
            elif ss == 'E':
                ss_type = 'sheet'

            ss_counts[ss_type] += 1

            if ss_type != 'loop':
                # calculate secondary structure normalised pLDDT
                avg_plddt_ss = sum(atom.bfactor for atom in residue) / len(residue)
                plddts_ss.append(avg_plddt_ss)

            if residue_id in interacting_residues:
                ss_interface_counts[ss_type] += 1

                # calculate interface pLDDT
                avg_plddt_residue = sum(atom.bfactor for atom in residue) / len(residue)
                plddts_interface.append(avg_plddt_residue)

    # Calculate percentages
    total_residues = sum(ss_counts.values())
    total_interface_residues = sum(ss_interface_counts.values())

    percentages = calculate_percentages(total_residues, ss_counts['helix'], ss_counts['sheet'])
    interface_percentages = calculate_percentages(total_interface_residues, ss_interface_counts['helix'], ss_interface_counts['sheet'])

    i_plddt = round(sum(plddts_interface) / len(plddts_interface) / 100, 2) if plddts_interface else 0
    ss_plddt = round(sum(plddts_ss) / len(plddts_ss) / 100, 2) if plddts_ss else 0

    return (*percentages, *interface_percentages, i_plddt, ss_plddt)

def calculate_percentages(total, helix, sheet):
    helix_percentage = round((helix / total) * 100,2) if total > 0 else 0
    sheet_percentage = round((sheet / total) * 100,2) if total > 0 else 0
    loop_percentage = round(((total - helix - sheet) / total) * 100,2) if total > 0 else 0

    return helix_percentage, sheet_percentage, loop_percentage

In [ ]:
#============================================================================
# Generic Utils
#============================================================================
# Inlined verbatim from functions/generic_utils.py; imports hoisted to the cell above.

# Define labels for dataframes
def generate_dataframe_labels():
    # labels for trajectory
    trajectory_labels = ['Design', 'Protocol', 'Length', 'Seed', 'Helicity', 'Target_Hotspot', 'Sequence', 'InterfaceResidues', 'pLDDT', 'pTM', 'i_pTM', 'pAE', 'i_pAE', 'i_pLDDT', 'ss_pLDDT', 'Unrelaxed_Clashes',
                        'Relaxed_Clashes', 'Binder_Energy_Score', 'Surface_Hydrophobicity', 'ShapeComplementarity', 'PackStat', 'dG', 'dSASA', 'dG/dSASA', 'Interface_SASA_%', 'Interface_Hydrophobicity', 'n_InterfaceResidues',
                        'n_InterfaceHbonds', 'InterfaceHbondsPercentage', 'n_InterfaceUnsatHbonds', 'InterfaceUnsatHbondsPercentage', 'Interface_Helix%', 'Interface_BetaSheet%', 'Interface_Loop%',
                        'Binder_Helix%', 'Binder_BetaSheet%', 'Binder_Loop%', 'InterfaceAAs', 'Target_RMSD', 'TrajectoryTime', 'Notes', 'TargetSettings', 'Filters', 'AdvancedSettings']

    # labels for mpnn designs
    core_labels = ['pLDDT', 'pTM', 'i_pTM', 'pAE', 'i_pAE', 'i_pLDDT', 'ss_pLDDT', 'Unrelaxed_Clashes', 'Relaxed_Clashes', 'Binder_Energy_Score', 'Surface_Hydrophobicity',
                    'ShapeComplementarity', 'PackStat', 'dG', 'dSASA', 'dG/dSASA', 'Interface_SASA_%', 'Interface_Hydrophobicity', 'n_InterfaceResidues', 'n_InterfaceHbonds', 'InterfaceHbondsPercentage',
                    'n_InterfaceUnsatHbonds', 'InterfaceUnsatHbondsPercentage', 'Interface_Helix%', 'Interface_BetaSheet%', 'Interface_Loop%', 'Binder_Helix%', 
                    'Binder_BetaSheet%', 'Binder_Loop%', 'InterfaceAAs', 'Hotspot_RMSD', 'Target_RMSD', 'Binder_pLDDT', 'Binder_pTM', 'Binder_pAE', 'Binder_RMSD']

    design_labels = ['Design', 'Protocol', 'Length', 'Seed', 'Helicity', 'Target_Hotspot', 'Sequence', 'InterfaceResidues', 'MPNN_score', 'MPNN_seq_recovery']

    for label in core_labels:
        design_labels += ['Average_' + label] + [f'{i}_{label}' for i in range(1, 6)]

    design_labels += ['DesignTime', 'Notes', 'TargetSettings', 'Filters', 'AdvancedSettings']

    final_labels = ['Rank'] + design_labels

    return trajectory_labels, design_labels, final_labels

# Create base directions of the project
def generate_directories(design_path):
    design_path_names = ["Accepted", "Accepted/Ranked", "Accepted/Animation", "Accepted/Plots", "Accepted/Pickle", "Trajectory",
                        "Trajectory/Relaxed", "Trajectory/Plots", "Trajectory/Clashing", "Trajectory/LowConfidence", "Trajectory/Animation",
                        "MPNN", "MPNN/Binder", "MPNN/Sequences", "MPNN/Relaxed", "Rejected"]
    design_paths = {}

    # make directories and set design_paths[FOLDER_NAME] variable
    for name in design_path_names:
        path = os.path.join(design_path, name)
        os.makedirs(path, exist_ok=True)
        design_paths[name] = path

    return design_paths

# generate CSV file for tracking designs not passing filters
def generate_filter_pass_csv(failure_csv, filter_json):
    if not os.path.exists(failure_csv):
        with open(filter_json, 'r') as file:
            data = json.load(file)
        
        # Create a list of modified keys
        names = ['Trajectory_logits_pLDDT', 'Trajectory_softmax_pLDDT', 'Trajectory_one-hot_pLDDT', 'Trajectory_final_pLDDT', 'Trajectory_Contacts', 'Trajectory_Clashes', 'Trajectory_WrongHotspot']
        special_prefixes = ('Average_', '1_', '2_', '3_', '4_', '5_')
        tracked_filters = set()

        for key in data.keys():
            processed_name = key  # Use the full key by default

            # Check if the key starts with any special prefixes
            for prefix in special_prefixes:
                if key.startswith(prefix):
                    # Strip the prefix and use the remaining part
                    processed_name = key.split('_', 1)[1]
                    break

            # Handle 'InterfaceAAs' with appending amino acids
            if 'InterfaceAAs' in processed_name:
                # Generate 20 variations of 'InterfaceAAs' with amino acids appended
                amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
                for aa in amino_acids:
                    variant_name = f"InterfaceAAs_{aa}"
                    if variant_name not in tracked_filters:
                        names.append(variant_name)
                        tracked_filters.add(variant_name)
            elif processed_name not in tracked_filters:
                # Add processed name if it hasn't been added before
                names.append(processed_name)
                tracked_filters.add(processed_name)

        # make dataframe with 0s
        df = pd.DataFrame(columns=names)
        df.loc[0] = [0] * len(names)

        df.to_csv(failure_csv, index=False)

# update failure rates from trajectories and early predictions
def update_failures(failure_csv, failure_column_or_dict):
    failure_df = pd.read_csv(failure_csv)
    
    def strip_model_prefix(name):
        # Strips the model-specific prefix if it exists
        parts = name.split('_')
        if parts[0].isdigit():
            return '_'.join(parts[1:])
        return name
    
    # update dictionary coming from complex prediction
    if isinstance(failure_column_or_dict, dict):
        # Update using a dictionary of failures
        for filter_name, count in failure_column_or_dict.items():
            stripped_name = strip_model_prefix(filter_name)
            if stripped_name in failure_df.columns:
                failure_df[stripped_name] += count
            else:
                failure_df[stripped_name] = count
    else:
        # Update a single column from trajectory generation
        failure_column = strip_model_prefix(failure_column_or_dict)
        if failure_column in failure_df.columns:
            failure_df[failure_column] += 1
        else:
            failure_df[failure_column] = 1
    
    failure_df.to_csv(failure_csv, index=False)

# Check if number of trajectories generated
def check_n_trajectories(design_paths, advanced_settings):
    n_trajectories = [f for f in os.listdir(design_paths["Trajectory/Relaxed"]) if f.endswith('.pdb')]

    if advanced_settings["max_trajectories"] is not False and len(n_trajectories) >= advanced_settings["max_trajectories"]:
        print(f"Target number of {str(len(n_trajectories))} trajectories reached, stopping execution...")
        return True
    else:
        return False
    
# Check if we have required number of accepted targets
def check_accepted_designs_dual(design_paths, species_csv_files, final_labels, advanced_settings, target_settings, design_labels):
    accepted_binders = [f for f in os.listdir(design_paths["Accepted"]) if f.endswith('.pdb')]
    number_accepted_binder = len(accepted_binders)

    if len(target_settings.get("species", [])) > 1:
        unique_entries = {f.split("model")[0] for f in accepted_binders}
        number_accepted_binder = len(unique_entries)

    if number_accepted_binder >= target_settings["number_of_final_designs"]:
        print(f"Target number {str(number_accepted_binder)} of designs reached! Reranking...")

        for species in target_settings["species"]:
            species_reached = rank_targets(
                accepted_binders,
                design_paths,
                species_csv_files[species]["mpnn_csv"],
                final_labels,
                species_csv_files[species]["final_csv"],
                advanced_settings,
                target_settings,
                design_labels,
            )
        return True
    else: 
        return False

# Check if we have required number of accepted targets
def check_accepted_designs_multi(design_paths, template_information, general_information, final_labels, advanced_settings, target_settings, design_labels):
    accepted_binders = [f for f in os.listdir(design_paths["Accepted"]) if f.endswith('.pdb')]
    number_accepted_binder = len(accepted_binders)

    if len(template_information) > 1:
        unique_entries = {f.split("model")[0] for f in accepted_binders}
        number_accepted_binder = len(unique_entries)

    if number_accepted_binder >= general_information["number_of_final_designs"]:
        print(f"Target number {str(number_accepted_binder)} of designs reached! Reranking...")

        for template in template_information:
            rank_targets(
                accepted_binders,
                design_paths,
                template_information[template]["mpnn_csv"],
                final_labels,
                template_information[template]["final_csv"],
                advanced_settings,
                target_settings,
                design_labels,
            )
        return True
    else: 
        return False



# For accepted target rank them, and analyse sequence and structure properties
def rank_targets(accepted_binders, design_paths, mpnn_csv, final_labels, final_csv, advanced_settings, target_settings, design_labels):
    # Clear the Ranked folder
    for f in os.listdir(design_paths["Accepted/Ranked"]):
        os.remove(os.path.join(design_paths["Accepted/Ranked"], f))

    # Load and rank designs
    design_df = pd.read_csv(mpnn_csv)
    design_df = design_df.sort_values('Average_i_pTM', ascending=False)
    final_df = pd.DataFrame(columns=final_labels)
    
    rank = 1
    for _, row in design_df.iterrows():
        for binder in accepted_binders:
            target_settings["binder_name"], model = binder.rsplit('_model', 1)
            if target_settings["binder_name"] == row['Design']:
                row_data = {'Rank': rank, **{label: row[label] for label in design_labels}}
                final_df = pd.concat([final_df, pd.DataFrame([row_data])], ignore_index=True)
                old_path = os.path.join(design_paths["Accepted"], binder)
                new_path = os.path.join(
                    design_paths["Accepted/Ranked"],
                    f"{rank}_{target_settings['binder_name']}_model{model.rsplit('.', 1)[0]}.pdb",
                )
                shutil.copyfile(old_path, new_path)
                rank += 1
                break

    # Save final ranked designs to CSV
    final_df.to_csv(final_csv, index=False)

    # Zip large folders if enabled in settings
    if advanced_settings["zip_animations"]:
        zip_and_empty_folder(design_paths["Trajectory/Animation"], '.html')
    if advanced_settings["zip_plots"]:
        zip_and_empty_folder(design_paths["Trajectory/Plots"], '.png')
    
    return 0

# Check if we have required number of accepted targets, rank them, and analyse sequence and structure properties
def check_accepted_designs(design_paths, mpnn_csv, final_labels, final_csv, advanced_settings, target_settings, design_labels):
    accepted_binders = [f for f in os.listdir(design_paths["Accepted"]) if f.endswith('.pdb')]

    if len(accepted_binders) >= target_settings["number_of_final_designs"]:
        print(f"Target number {str(len(accepted_binders))} of designs reached! Reranking...")

        # clear the Ranked folder in case we added new designs in the meantime so we rerank them all
        for f in os.listdir(design_paths["Accepted/Ranked"]):
            os.remove(os.path.join(design_paths["Accepted/Ranked"], f))

        # load dataframe of designed binders
        design_df = pd.read_csv(mpnn_csv)
        design_df = design_df.sort_values('Average_i_pTM', ascending=False)
        
        # create final csv dataframe to copy matched rows, initialize with the column labels
        final_df = pd.DataFrame(columns=final_labels)

        # check the ranking of the designs and copy them with new ranked IDs to the folder
        rank = 1
        for _, row in design_df.iterrows():
            for binder in accepted_binders:
                target_settings["binder_name"], model = binder.rsplit('_model', 1)
                if target_settings["binder_name"] == row['Design']:
                    # rank and copy into ranked folder
                    row_data = {'Rank': rank, **{label: row[label] for label in design_labels}}
                    final_df = pd.concat([final_df, pd.DataFrame([row_data])], ignore_index=True)
                    old_path = os.path.join(design_paths["Accepted"], binder)
                    new_path = os.path.join(design_paths["Accepted/Ranked"], f"{rank}_{target_settings['binder_name']}_model{model.rsplit('.', 1)[0]}.pdb")
                    shutil.copyfile(old_path, new_path)

                    rank += 1
                    break

        # save the final_df to final_csv
        final_df.to_csv(final_csv, index=False)

        # zip large folders to save space
        if advanced_settings["zip_animations"]:
            zip_and_empty_folder(design_paths["Trajectory/Animation"], '.html')

        if advanced_settings["zip_plots"]:
            zip_and_empty_folder(design_paths["Trajectory/Plots"], '.png')

        return True

    else:
        return False

# Load required helicity value
def load_helicity(advanced_settings):
    if advanced_settings["random_helicity"] is True:
        # will sample a random bias towards helicity
        helicity_value = round(np.random.uniform(-3, 1),2)
    elif advanced_settings["weights_helicity"] != 0:
        # using a preset helicity bias
        helicity_value = advanced_settings["weights_helicity"]
    else:
        # no bias towards helicity
        helicity_value = 0
    return helicity_value

# Report JAX-capable devices
def check_jax_gpu():
    devices = jax.devices()

    has_gpu = any(device.platform == 'gpu' for device in devices)

    if not has_gpu:
        print("No GPU device found, terminating.")
        exit()
    else:
        print("Available GPUs:")
        for i, device in enumerate(devices):
            print(f"{device.device_kind}{i + 1}: {device.platform}")

# check all input files being passed
def perform_input_check(args):
    # Get the directory of the current script
    # was os.path.dirname(os.path.abspath(__file__)); there is no __file__ in a
    # notebook cell, and PROJECT_ROOT is the same directory the package sat in
    binder_script_path = str(PROJECT_ROOT)

    # Ensure settings file is provided
    if not args.settings:
        print("Error: --settings is required.")
        exit()

    # # Set default filters.json path if not provided
    # if not args.filters:
    #     args.filters = os.path.join(binder_script_path, 'settings_filters', 'default_filters.json')

    # Set a random advanced json settings file if not provided
    if not args.advanced:
        args.advanced = os.path.join(binder_script_path, 'settings_advanced', 'default_4stage_multimer.json')

    # return args.settings, args.filters, args.advanced
    return args.settings, args.advanced

# check specific advanced settings
def perform_advanced_settings_check(advanced_settings, bindcraft_folder):
    # set paths to model weights and executables
    if bindcraft_folder == "colab":
        advanced_settings["af_params_dir"] = '/content/bindcraft/params/'
        advanced_settings["dssp_path"] = '/content/bindcraft/functions/dssp'
        advanced_settings["dalphaball_path"] = '/content/bindcraft/functions/DAlphaBall.gcc'
    else:
        # Set paths individually if they are not already set
        if not advanced_settings["af_params_dir"]:
            advanced_settings["af_params_dir"] = bindcraft_folder
        if not advanced_settings["dssp_path"]:
            advanced_settings["dssp_path"] = os.path.join(bindcraft_folder, 'functions', 'dssp')
        if not advanced_settings["dalphaball_path"]:
            advanced_settings["dalphaball_path"] = os.path.join(bindcraft_folder, 'functions', 'DAlphaBall.gcc')

    # check formatting of omit_AAs setting
        omit_aas = advanced_settings["omit_AAs"]
    if advanced_settings["omit_AAs"] in [None, False, '']:
        advanced_settings["omit_AAs"] = None
    elif isinstance(advanced_settings["omit_AAs"], str):
        advanced_settings["omit_AAs"] = advanced_settings["omit_AAs"].strip()

    return advanced_settings

# Load settings from JSONs
def load_json_settings(settings_json, advanced_json):
    # load settings from json files
    with open(settings_json, 'r') as file:
        target_settings = json.load(file)

    with open(advanced_json, 'r') as file:
        advanced_settings = json.load(file)

    # with open(filters_json, 'r') as file:
    #     filters = json.load(file)

    return target_settings, advanced_settings

# AF2 model settings, make sure non-overlapping models with template option are being used for design and re-prediction
def load_af2_models(af_multimer_setting):
    if af_multimer_setting:
        design_models = [0,1,2,3,4]
        prediction_models = [0,1]
        multimer_validation = False
    else:
        design_models = [0,1]
        prediction_models = [0,1,2,3,4]
        multimer_validation = True

    return design_models, prediction_models, multimer_validation

# create csv for insertion of data
def create_dataframe(csv_file, columns):
    if not os.path.exists(csv_file):
        df = pd.DataFrame(columns=columns)
        df.to_csv(csv_file, index=False)

# insert row of statistics into csv
def insert_data(csv_file, data_array):
    df = pd.DataFrame([data_array])
    df.to_csv(csv_file, mode='a', header=False, index=False)

# save generated sequence
def save_fasta(design_name, sequence, design_paths):
    fasta_path = os.path.join(design_paths["MPNN/Sequences"], design_name+".fasta")
    with open(fasta_path,"w") as fasta:
        line = f'>{design_name}\n{sequence}'
        fasta.write(line+"\n")

# clean unnecessary rosetta information from PDB
def clean_pdb(pdb_file):
    # Read the pdb file and filter relevant lines
    with open(pdb_file, 'r') as f_in:
        relevant_lines = [line for line in f_in if line.startswith(('ATOM', 'HETATM', 'MODEL', 'TER', 'END'))]

    # Write the cleaned lines back to the original pdb file
    with open(pdb_file, 'w') as f_out:
        f_out.writelines(relevant_lines)

def zip_and_empty_folder(folder_path, extension):
    folder_basename = os.path.basename(folder_path)
    zip_filename = os.path.join(os.path.dirname(folder_path), folder_basename + '.zip')

    # Open the zip file in 'a' mode to append if it exists, otherwise create a new one
    with zipfile.ZipFile(zip_filename, 'a', zipfile.ZIP_DEFLATED) as zipf:
        for file in os.listdir(folder_path):
            if file.endswith(extension):
                # Create an absolute path
                file_path = os.path.join(folder_path, file)
                # Add file to zip file, replacing it if it already exists
                zipf.write(file_path, arcname=file)
                # Remove the file after adding it to the zip
                os.remove(file_path)
    print(f"Files in folder '{folder_path}' have been zipped and removed.")

# calculate averages for statistics
def calculate_averages(statistics, handle_aa=False):
    # Initialize a dictionary to hold the sums of each statistic
    sums = {}
    # Initialize a dictionary to hold the sums of each amino acid count
    aa_sums = {}

    # Iterate over the model numbers
    for model_num in range(1, 6):  # assumes models are numbered 1 through 5
        # Check if the model's data exists
        if model_num in statistics:
            # Get the model's statistics
            model_stats = statistics[model_num]
            # For each statistic, add its value to the sum
            for stat, value in model_stats.items():
                # If this is the first time we've seen this statistic, initialize its sum to 0
                if stat not in sums:
                    sums[stat] = 0

                if value is None:
                    value = 0

                # If the statistic is mpnn_interface_AA and we're supposed to handle it separately, do so
                if handle_aa and stat == 'InterfaceAAs':
                    for aa, count in value.items():
                        # If this is the first time we've seen this amino acid, initialize its sum to 0
                        if aa not in aa_sums:
                            aa_sums[aa] = 0
                        aa_sums[aa] += count
                else:
                    sums[stat] += value

    # Now that we have the sums, we can calculate the averages
    averages = {stat: round(total / len(statistics), 2) for stat, total in sums.items()}

    # If we're handling aa counts, calculate their averages
    if handle_aa:
        aa_averages = {aa: round(total / len(statistics),2) for aa, total in aa_sums.items()}
        averages['InterfaceAAs'] = aa_averages

    return averages

# filter designs based on feature thresholds
def check_filters(mpnn_data, design_labels, filters):
    # check mpnn_data against labels
    mpnn_dict = {label: value for label, value in zip(design_labels, mpnn_data)}

    unmet_conditions = []

    # check filters against thresholds
    for label, conditions in filters.items():
        # special conditions for interface amino acid counts
        if label == 'Average_InterfaceAAs' or label == '1_InterfaceAAs' or label == '2_InterfaceAAs' or label == '3_InterfaceAAs' or label == '4_InterfaceAAs' or label == '5_InterfaceAAs':
            for aa, aa_conditions in conditions.items():
                if mpnn_dict.get(label) is None:
                    continue
                value = mpnn_dict.get(label).get(aa)
                if value is None or aa_conditions["threshold"] is None:
                    continue
                if aa_conditions["higher"]:
                    if value < aa_conditions["threshold"]:
                        unmet_conditions.append(f"{label}_{aa}")
                else:
                    if value > aa_conditions["threshold"]:
                        unmet_conditions.append(f"{label}_{aa}")
        else:
            # if no threshold, then skip
            value = mpnn_dict.get(label)
            if value is None or conditions["threshold"] is None:
                continue
            if conditions["higher"]:
                if value < conditions["threshold"]:
                    unmet_conditions.append(label)
            else:
                if value > conditions["threshold"]:
                    unmet_conditions.append(label)

    # if all filters are passed then return True
    if len(unmet_conditions) == 0:
        return True
    # if some filters were unmet, print them out
    else:
        return unmet_conditions

def generate_biased_sequence(motif_part, total_length):
    # Constants
    motif_length = len(motif_part)
    # Calculate the length of the repeat sequence including the motif
    rest_length= total_length-motif_length
    
    if total_length < motif_length:
        raise ValueError("Total length is too short to accommodate motif")

    # Randomly distribute the remaining length on both sides, while ensuring the minimum 10 residues
    left_side_length = random.randint(0, rest_length)
    right_side_length = rest_length - left_side_length

    # Generate the sequence parts
    left_side = 'X' * left_side_length
    right_side = 'X' * right_side_length

    # Generate the full sequence
    full_sequence = left_side + motif_part + right_side
    
    return full_sequence


def generate_darpin_sequence(n_terminus, c_terminus, spacer_sequence, loop_length, repeats, target_length):
    # Length of fixed sequences
    n_terminus_len = len(n_terminus)
    c_terminus_len = len(c_terminus)
    spacer_len = len(spacer_sequence) * repeats
    
    # Calculate the length of fixed sections
    fixed_length = n_terminus_len + c_terminus_len + spacer_len
    
    # Calculate the remaining length for the loops
    remaining_length = target_length - fixed_length
    
    if remaining_length < 0:
        raise ValueError("Target length is too short to accommodate the fixed sequences.")
    
    # Calculate the number of characters to allocate to the loops
    loop_total_length = remaining_length
    
    # Distribute the remaining length randomly across the loops
    loop_lengths = distribute_loop_length(loop_total_length, repeats)

    # Generate the full sequence
    full_sequence = n_terminus
    
    # Add the loops and spacer sequence
    for i in range(repeats):
        full_sequence += "X" * loop_lengths[i] + spacer_sequence
    
    # Add the C-terminus
    full_sequence += c_terminus
    
    # Return the full sequence and its length
    return full_sequence, len(full_sequence)


def distribute_loop_length(total_length, num_loops):
    """Randomly distribute the total length across num_loops"""
    # Generate random loop lengths and make sure they sum to total_length
    loop_lengths = [random.randint(1, total_length // num_loops) for _ in range(num_loops)]
    total_assigned = sum(loop_lengths)
    
    # Adjust the last loop length to make the total sum equal to total_length
    loop_lengths[-1] += (total_length - total_assigned)
    
    return loop_lengths

def parse_arguments():
    """Parse command-line arguments."""
    parser = argparse.ArgumentParser(description='Script to run BindCraft binder design.')
    parser.add_argument('--settings', '-s', type=str, required=True,
                        help='Path to the basic settings.json file. Required.')
    parser.add_argument('--filters', '-f', type=str, default='./settings_filters/default_filters.json',
                        help='Path to the filters.json file used to filter design. If not provided, default will be used.')
    parser.add_argument('--advanced', '-a', type=str, default='./settings_advanced/4stage_multimer.json',
                        help='Path to the advanced.json file with additional design settings. If not provided, default will be used.')
    return parser.parse_args()


def prepare_species_csv_files(target_settings, file_suffixes, args, design_labels, final_labels, trajectory_labels, species):
    """Prepare output CSV files for each species."""
    if not species:
        raise ValueError("No species defined in the target settings.")
    
    species_csv_files = {}
    for sp in species:
        species_csv_files[sp] = {
            key: os.path.join(target_settings["design_path"], value.format(sp))
            for key, value in file_suffixes.items()
        }
        create_dataframe(species_csv_files[sp]["mpnn_csv"], design_labels)
        create_dataframe(species_csv_files[sp]["final_csv"], final_labels)
        create_dataframe(species_csv_files[sp]["trajectory_csv"], trajectory_labels)
        generate_filter_pass_csv(species_csv_files[sp]["failure_csv"], args.filters)
    return species_csv_files


def check_dependencies(bindcraft_folder, advanced_settings):
    """Check for necessary dependencies and permissions."""
    if not os.path.exists(os.path.join(bindcraft_folder, "params")):
        raise FileNotFoundError(
            f"The 'params' folder is missing in '{bindcraft_folder}'. "
            f"Please ensure you have a copy of the AlphaFold dependencies in the BindCraft folder."
        )
    advanced_settings["af_params_dir"] = bindcraft_folder
    advanced_settings["dssp_path"] = os.path.join(bindcraft_folder, 'functions/dssp')
    advanced_settings["dalphaball_path"] = os.path.join(bindcraft_folder, 'functions/DAlphaBall.gcc')
    for path in [advanced_settings["dssp_path"], advanced_settings["dalphaball_path"]]:
        if not os.access(path, os.X_OK):  
            raise PermissionError(
                f"Execute permission is missing for: {path}. "
                f"Ensure the file has execute permissions using 'chmod +x {path}'."
            )

def average_trajectory_dataframes(dataframes):
    """
    Merges multiple DataFrames by:
    - Keeping the first occurrence of `keep_one_copy` columns.
    - Dropping `drop_columns`.
    - Computing the mean for `mean_columns`.
    - Merging `InterfaceResidues` as a unique set of values.

    Args:
        dataframes (list of pd.DataFrame): List of DataFrames to merge.

    Returns:
        pd.DataFrame: A single row DataFrame with merged values.
    """
    keep_one_copy = ['Design', 'Protocol', 'Length', 'Seed', 'Sequence', 'TrajectoryTime', 'Notes', 'TargetSettings', 'Filters', 'AdvancedSettings']
    drop_columns = ['Target_Hotspot', 'InterfaceAAs']
    mean_columns = ['Helicity',  'pLDDT', 'pTM', 'i_pTM', 'pAE', 'i_pAE', 'i_pLDDT', 'ss_pLDDT', 'Unrelaxed_Clashes', 'Relaxed_Clashes', 'Binder_Energy_Score', 'Surface_Hydrophobicity', 
                    'ShapeComplementarity', 'PackStat', 'dG', 'dSASA', 'dG/dSASA', 'Interface_SASA_%', 'Interface_Hydrophobicity', 
                    'n_InterfaceResidues', 'n_InterfaceHbonds', 'InterfaceHbondsPercentage', 'n_InterfaceUnsatHbonds', 
                    'InterfaceUnsatHbondsPercentage', 'Interface_Helix%', 'Interface_BetaSheet%', 'Interface_Loop%', 
                    'Binder_Helix%', 'Binder_BetaSheet%', 'Binder_Loop%']
    set_columns = ['InterfaceResidues']

    combined_df = pd.concat(dataframes, ignore_index=True)
    combined_result = combined_df[keep_one_copy].iloc[0].to_dict()
    for col in mean_columns:
        combined_result[col] = combined_df[col].mean()

    combined_result['InterfaceResidues'] = ",".join(
        sorted(set(
            residue.strip()  # Remove any accidental whitespace
            for residues in combined_df['InterfaceResidues'].dropna()  # Remove NaN values
            for residue in residues.split(",")  # Flatten all lists of residues
        ))
    )

    return pd.DataFrame([combined_result])

# Prepare species-specific CSV files
CSV_SPECIES_SUFFIXES = {
    "mpnn_csv": "mpnn_design_stats_{}.csv",
    "final_csv": "final_design_stats_{}.csv",
    "failure_csv": "failure_csv_{}.csv",
    "trajectory_csv": "trajectory_{}.csv",
}

def collect_target_information(bindcraft_path, target_settings, design_labels, final_labels, trajectory_labels, args):  
    binder_information = {}
    binder_information.update(target_settings["general_information"])

    print("loading template configurations")
    template_information = {}
    number_of_templates = len(target_settings) - 1
    for current_template_information in target_settings:
        if re.match(r"^template_.*_information$", current_template_information):
            template_information[current_template_information] = target_settings[current_template_information]
            if not template_information[current_template_information]["template_name"]:
                raise ValueError("Have you forgot to name your template in the target settings?")
            
            for key, value in CSV_SPECIES_SUFFIXES.items():
                template_information[current_template_information][key] = os.path.join(
                    binder_information["design_path"],
                    value.format(template_information[current_template_information]["template_name"])
                )
            if not template_information.get(current_template_information, {}).get("template_targeting_strategy", ""):
                template_information[current_template_information]["template_targeting_strategy"] = "target"

            if not template_information.get(current_template_information, {}).get("template_mpnn_filters", ""):
                if number_of_templates == 1:
                    template_information[current_template_information]["template_mpnn_filters"] = os.path.join(
                        f"{bindcraft_path}/settings_filters/", "default_mono_filters.json"
                        )
                    current_filter_file_path = f"{template_information[current_template_information]['template_mpnn_filters_path']}"
                    with open(current_filter_file_path, 'r') as file:
                        current_filters = json.load(file)
                    template_information[current_template_information]["template_mpnn_filters"] = current_filters

                else:
                    # template_information[current_template_information]["template_mpnn_filters_path"] = os.path.join(
                    #     f"{bindcraft_path}/settings_filters/", f"default_multi_{template_information[current_template_information]['template_targeting_strategy']}_filters.json"
                    #     )
                    # print(template_information[current_template_information])
                    current_filter_file_path = f"{template_information[current_template_information]['template_mpnn_filters_path']}"
                    with open(current_filter_file_path, 'r') as file:
                        current_filters = json.load(file)
                    template_information[current_template_information]["template_mpnn_filters"] = current_filters
                    # print(template_information[current_template_information]["template_targeting_strategy"])
                    # print(current_filters)
            
            create_dataframe(template_information[current_template_information]["mpnn_csv"], design_labels)
            create_dataframe(template_information[current_template_information]["final_csv"], final_labels)
            create_dataframe(template_information[current_template_information]["trajectory_csv"], trajectory_labels)
            generate_filter_pass_csv(template_information[current_template_information]["failure_csv"], template_information[current_template_information]['template_mpnn_filters_path'])
    
    return binder_information, template_information

In [ ]:
#============================================================================
# PyRosetta Utils
#============================================================================
# Inlined verbatim from functions/pyrosetta_utils.py; imports hoisted to the cell above.

# Rosetta interface scores
def score_interface(pdb_file, binder_chain="B"):
    # load pose
    pose = pr.pose_from_pdb(pdb_file)

    # analyze interface statistics
    iam = InterfaceAnalyzerMover()
    interface = "A_B"
    docking_partners_type = getattr(pr.rosetta.core.pose, "DockingPartners", None)
    if docking_partners_type is not None:
        interface = docking_partners_type.docking_partners_from_string(interface)
    iam.set_interface(interface)
    scorefxn = pr.get_fa_scorefxn()
    iam.set_scorefunction(scorefxn)
    iam.set_compute_packstat(True)
    iam.set_compute_interface_energy(True)
    iam.set_calc_dSASA(True)
    iam.set_calc_hbond_sasaE(True)
    iam.set_compute_interface_sc(True)
    iam.set_pack_separated(True)
    iam.apply(pose)

    # Initialize dictionary with all amino acids
    interface_AA = {aa: 0 for aa in 'ACDEFGHIKLMNPQRSTVWY'}

    # Initialize list to store PDB residue IDs at the interface
    interface_residues_set = hotspot_residues(pdb_file, binder_chain)
    interface_residues_pdb_ids = []
    
    # Iterate over the interface residues
    for pdb_res_num, aa_type in interface_residues_set.items():
        # Increase the count for this amino acid type
        interface_AA[aa_type] += 1

        # Append the binder_chain and the PDB residue number to the list
        interface_residues_pdb_ids.append(f"{binder_chain}{pdb_res_num}")

    # count interface residues
    interface_nres = len(interface_residues_pdb_ids)

    # Convert the list into a comma-separated string
    interface_residues_pdb_ids_str = ','.join(interface_residues_pdb_ids)

    # Calculate the percentage of hydrophobic residues at the interface of the binder
    hydrophobic_aa = set('ACFILMPVWY')
    hydrophobic_count = sum(interface_AA[aa] for aa in hydrophobic_aa)
    if interface_nres != 0:
        interface_hydrophobicity = (hydrophobic_count / interface_nres) * 100
    else:
        interface_hydrophobicity = 0

    # retrieve statistics
    interfacescore = iam.get_all_data()
    interface_sc = interfacescore.sc_value # shape complementarity
    interface_interface_hbonds = interfacescore.interface_hbonds # number of interface H-bonds
    interface_dG = iam.get_interface_dG() # interface dG
    interface_dSASA = iam.get_interface_delta_sasa() # interface dSASA (interface surface area)
    interface_packstat = iam.get_interface_packstat() # interface pack stat score
    interface_dG_SASA_ratio = interfacescore.dG_dSASA_ratio * 100 # ratio of dG/dSASA (normalised energy for interface area size)
    buns_filter = XmlObjects.static_get_filter('<BuriedUnsatHbonds report_all_heavy_atom_unsats="true" scorefxn="scorefxn" ignore_surface_res="false" use_ddG_style="true" dalphaball_sasa="1" probe_radius="1.1" burial_cutoff_apo="0.2" confidence="0" />')
    interface_delta_unsat_hbonds = buns_filter.report_sm(pose)

    if interface_nres != 0:
        interface_hbond_percentage = (interface_interface_hbonds / interface_nres) * 100 # Hbonds per interface size percentage
        interface_bunsch_percentage = (interface_delta_unsat_hbonds / interface_nres) * 100 # Unsaturated H-bonds per percentage
    else:
        interface_hbond_percentage = None
        interface_bunsch_percentage = None

    # calculate binder energy score
    chain_design = ChainSelector(binder_chain)
    tem = pr.rosetta.core.simple_metrics.metrics.TotalEnergyMetric()
    tem.set_scorefunction(scorefxn)
    tem.set_residue_selector(chain_design)
    binder_score = tem.calculate(pose)

    # calculate binder SASA fraction
    bsasa = pr.rosetta.core.simple_metrics.metrics.SasaMetric()
    bsasa.set_residue_selector(chain_design)
    binder_sasa = bsasa.calculate(pose)

    if binder_sasa > 0:
        interface_binder_fraction = (interface_dSASA / binder_sasa) * 100
    else:
        interface_binder_fraction = 0

    # calculate surface hydrophobicity
    binder_pose = {pose.pdb_info().chain(pose.conformation().chain_begin(i)): p for i, p in zip(range(1, pose.num_chains()+1), pose.split_by_chain())}[binder_chain]

    layer_sel = pr.rosetta.core.select.residue_selector.LayerSelector()
    layer_sel.set_layers(pick_core = False, pick_boundary = False, pick_surface = True)
    surface_res = layer_sel.apply(binder_pose)

    exp_apol_count = 0
    total_count = 0 
    
    # count apolar and aromatic residues at the surface
    for i in range(1, len(surface_res) + 1):
        if surface_res[i] == True:
            res = binder_pose.residue(i)

            # count apolar and aromatic residues as hydrophobic
            if res.is_apolar() == True or res.name() == 'PHE' or res.name() == 'TRP' or res.name() == 'TYR':
                exp_apol_count += 1
            total_count += 1

    surface_hydrophobicity = exp_apol_count/total_count

    # output interface score array and amino acid counts at the interface
    interface_scores = {
    'binder_score': binder_score,
    'surface_hydrophobicity': surface_hydrophobicity,
    'interface_sc': interface_sc,
    'interface_packstat': interface_packstat,
    'interface_dG': interface_dG,
    'interface_dSASA': interface_dSASA,
    'interface_dG_SASA_ratio': interface_dG_SASA_ratio,
    'interface_fraction': interface_binder_fraction,
    'interface_hydrophobicity': interface_hydrophobicity,
    'interface_nres': interface_nres,
    'interface_interface_hbonds': interface_interface_hbonds,
    'interface_hbond_percentage': interface_hbond_percentage,
    'interface_delta_unsat_hbonds': interface_delta_unsat_hbonds,
    'interface_delta_unsat_hbonds_percentage': interface_bunsch_percentage
    }

    # round to two decimal places
    interface_scores = {k: round(v, 2) if isinstance(v, float) else v for k, v in interface_scores.items()}

    return interface_scores, interface_AA, interface_residues_pdb_ids_str

# align pdbs to have same orientation
def align_pdbs(reference_pdb, align_pdb, reference_chain_id, align_chain_id):
    # initiate poses
    reference_pose = pr.pose_from_pdb(reference_pdb)
    align_pose = pr.pose_from_pdb(align_pdb)

    align = AlignChainMover()
    align.pose(reference_pose)

    # If the chain IDs contain commas, split them and only take the first value
    reference_chain_id = reference_chain_id.split(',')[0]
    align_chain_id = align_chain_id.split(',')[0]

    # Get the chain number corresponding to the chain ID in the poses
    reference_chain = pr.rosetta.core.pose.get_chain_id_from_chain(reference_chain_id, reference_pose)
    align_chain = pr.rosetta.core.pose.get_chain_id_from_chain(align_chain_id, align_pose)

    align.source_chain(align_chain)
    align.target_chain(reference_chain)
    align.apply(align_pose)

    # Overwrite aligned pdb
    align_pose.dump_pdb(align_pdb)
    clean_pdb(align_pdb)

# calculate the rmsd without alignment
def unaligned_rmsd(reference_pdb, align_pdb, reference_chain_id, align_chain_id):
    reference_pose = pr.pose_from_pdb(reference_pdb)
    align_pose = pr.pose_from_pdb(align_pdb)

    # Define chain selectors for the reference and align chains
    reference_chain_selector = ChainSelector(reference_chain_id)
    align_chain_selector = ChainSelector(align_chain_id)

    # Apply selectors to get residue subsets
    reference_chain_subset = reference_chain_selector.apply(reference_pose)
    align_chain_subset = align_chain_selector.apply(align_pose)

    # Convert subsets to residue index vectors
    reference_residue_indices = get_residues_from_subset(reference_chain_subset)
    align_residue_indices = get_residues_from_subset(align_chain_subset)

    # Create empty subposes
    reference_chain_pose = pr.Pose()
    align_chain_pose = pr.Pose()

    # Fill subposes
    pose_from_pose(reference_chain_pose, reference_pose, reference_residue_indices)
    pose_from_pose(align_chain_pose, align_pose, align_residue_indices)

    # Calculate RMSD using the RMSDMetric
    rmsd_metric = RMSDMetric()
    rmsd_metric.set_comparison_pose(reference_chain_pose)
    rmsd = rmsd_metric.calculate(align_chain_pose)

    return round(rmsd, 2)

# Relax designed structure
def pr_relax(pdb_file, relaxed_pdb_path):
    if not os.path.exists(relaxed_pdb_path):
        # Generate pose
        pose = pr.pose_from_pdb(pdb_file)
        start_pose = pose.clone()

        ### Generate movemaps
        mmf = MoveMap()
        mmf.set_chi(True) # enable sidechain movement
        mmf.set_bb(True) # enable backbone movement, can be disabled to increase speed by 30% but makes metrics look worse on average
        mmf.set_jump(False) # disable whole chain movement

        # Run FastRelax
        fastrelax = FastRelax()
        scorefxn = pr.get_fa_scorefxn()
        fastrelax.set_scorefxn(scorefxn)
        fastrelax.set_movemap(mmf) # set MoveMap
        fastrelax.max_iter(200) # default iterations is 2500
        fastrelax.min_type("lbfgs_armijo_nonmonotone")
        fastrelax.constrain_relax_to_start_coords(True)
        fastrelax.apply(pose)

        # Align relaxed structure to original trajectory
        align = AlignChainMover()
        align.source_chain(0)
        align.target_chain(0)
        align.pose(start_pose)
        align.apply(pose)

        # Copy B factors from start_pose to pose
        for resid in range(1, pose.total_residue() + 1):
            if pose.residue(resid).is_protein():
                # Get the B factor of the first heavy atom in the residue
                bfactor = start_pose.pdb_info().bfactor(resid, 1)
                for atom_id in range(1, pose.residue(resid).natoms() + 1):
                    pose.pdb_info().bfactor(resid, atom_id, bfactor)

        # output relaxed and aligned PDB
        pose.dump_pdb(relaxed_pdb_path)
        clean_pdb(relaxed_pdb_path)

In [ ]:
#============================================================================
# Multi ColabDesign Utils
#============================================================================
# Inlined verbatim from functions/multi_colabdesign_utils.py; imports hoisted to the cell above.

def binder_hallucination_multi(template_information, design_paths, design_name, advanced_settings, length, seed, chain, helicity_value, design_models):
    model_pdb_path = os.path.join(design_paths["Trajectory"], design_name+".pdb")
    # clear GPU memory for new trajectory
    clear_mem()
    passing = True

    print("updates")
    af_model_optimization = mk_afdesign_model(
        protocol="binder", 
        debug=False, 
        data_dir=advanced_settings["af_params_dir"], 
        use_multimer=advanced_settings["use_multimer_design"], 
        num_recycles=advanced_settings["num_recycles_design"],
        best_metric='loss'
    )

    # check for hotspots and initialize the model with corresponding hotsxpots:
    for template in template_information:
        if template_information[template]["template_hostspot_residues"] == "":
            template_information[template]["template_hostspot_residues"] = None
        af_model_optimization.prep_inputs(
                pdb_filename=template_information[template]["template_pdb"], 
                chain=chain, 
                binder_len=length, 
                hotspot=template_information[template]["template_hostspot_residues"], 
                seed=seed, 
                rm_aa=advanced_settings["omit_AAs"],
                # rm_template_ic=True
                )

        template_information[template]["batch"] = copy.deepcopy(af_model_optimization._inputs['batch'])
        template_information[template]["hotspot"] = copy.deepcopy(af_model_optimization.opt.get("hotspot"))
    
    ### Update weights based on specified settings
    af_model_optimization.opt["weights"].update({
        "pae":advanced_settings["weights_pae_intra"],
        "plddt":advanced_settings["weights_plddt"],
        "i_pae":advanced_settings["weights_pae_inter"],
        "con":advanced_settings["weights_con_intra"],
        "i_con":advanced_settings["weights_con_inter"],
    })

    # redefine intramolecular contacts (con) and intermolecular contacts (i_con) definitions
    af_model_optimization.opt["con"].update({
        "num":advanced_settings["intra_contact_number"],
        "cutoff":advanced_settings["intra_contact_distance"],
        "binary":False,
        "seqsep":9})
    af_model_optimization.opt["i_con"].update({
        "num":advanced_settings["inter_contact_number"],
        "cutoff":advanced_settings["inter_contact_distance"],
        "binary":False})

    ### additional loss functions: 
    # radius of gyration loss
    if advanced_settings["use_rg_loss"]:
        add_rg_loss(af_model_optimization, advanced_settings["weights_rg"])
    # interface pTM loss
    if advanced_settings["use_i_ptm_loss"]:
        add_i_ptm_loss(af_model_optimization, advanced_settings["weights_iptm"])
    # add target plddt loss 
    if advanced_settings["use_target_plddt_loss"]:
        add_target_plddt_loss(af_model_optimization, advanced_settings["weights_target_plddt"])
    # add the helicity loss
    add_helix_loss(af_model_optimization, helicity_value)


    # calculate the number of mutations to do based on the length of the protein
    greedy_tries = math.ceil(length * (advanced_settings["greedy_percentage"] / 100))

#SECTIO II-III run trajectory optimization
    ### start design algorithm based on selection
    if advanced_settings["design_algorithm"] == '2stage':
        # uses gradient descend to get a PSSM profile and then uses PSSM to bias the sampling of random mutations to decrease loss
        raise NotImplementedError("not yet implemented")

    elif advanced_settings["design_algorithm"] == '3stage':
        # 3 stage design using logits, softmax, and one hot encoding
        raise NotImplementedError("not yet implemented")

    elif advanced_settings["design_algorithm"] == 'greedy':
        # design by using random mutations that decrease loss
        raise NotImplementedError("not yet implemented")

    elif advanced_settings["design_algorithm"] == 'mcmc':
        # design by using random mutations that decrease loss
        raise NotImplementedError("not yet implemented")
    
    elif advanced_settings["design_algorithm"] == '4stage_duo':
        if len(template_information) == 0:
            print("Preparing Logits")
            raise NotImplementedError() 

        else:
            if advanced_settings["buffer_iterations"] > 0:
                print("Preparing Logits on one target trajectory")
                first_target_template_information = {k: v for k, v in [next(((k, v) for k, v in template_information.items() if v['template_targeting_strategy'] in ['', 'target']), (None, None))] if k}
                design_multi(af_model_optimization, first_target_template_information, advanced_settings, iters=advanced_settings["buffer_iterations"], e_soft=0, models=design_models, num_models=1, sample_models=advanced_settings["sample_models"], save_best=True, dropout=False, binder_length=length)
            
            if advanced_settings["use_loss_decay"]:
                print(f"Stage 1 adv.: Increasing the detarget loss over the next {advanced_settings['decay_iters']} iters")
                design_multi(af_model_optimization, template_information, advanced_settings, iters=advanced_settings["decay_iters"], e_soft=0, models=design_models, num_models=1, sample_models=advanced_settings["sample_models"], save_best=True, dropout=False, binder_length=length, decay_iters=advanced_settings["decay_iters"])
                
                print(f"Stage 1: Test Logits")
                design_multi(af_model_optimization, template_information, advanced_settings, iters=advanced_settings["test_iterations"], e_soft=0.9, models=design_models, num_models=1, sample_models=advanced_settings["sample_models"], save_best=True, dropout=False, binder_length=length, decay_scale=advanced_settings["default_detargeting_decay"])

            else:
                print("Stage 1: Test Logits")
                # optional raw-logits warm-up (e_soft=0, so soft stays 0 -- no softmax) before the soft ramp
                _warmup = int(advanced_settings.get("multi_stage1_warmup_iters", 0) or 0)
                if _warmup > 0:
                    design_multi(af_model_optimization, template_information, advanced_settings, iters=_warmup, e_soft=0, models=design_models, num_models=1, sample_models=advanced_settings["sample_models"], save_best=True, dropout=False, binder_length=length, decay_scale=advanced_settings["default_detargeting_decay"])
                design_multi(af_model_optimization, template_information, advanced_settings, iters=advanced_settings["test_iterations"], e_soft=0.9, models=design_models, num_models=1, sample_models=advanced_settings["sample_models"], save_best=True, dropout=False, binder_length=length, decay_scale=advanced_settings["default_detargeting_decay"])
            # determine pLDDT of best iteration according to lowest 'loss' value
            initial_plddt = get_best_plddt(af_model_optimization, length)

            # if best iteration has high enough confidence then continue
            if initial_plddt > 0.65:
                print("Initial trajectory pLDDT good, continuing: "+str(initial_plddt))
                if advanced_settings["optimise_beta"]:
                    # temporarily dump model to assess secondary structure
                    af_model_optimization.save_pdb(model_pdb_path)
                    _, beta, *_ = calc_ss_percentage(model_pdb_path, advanced_settings, 'B')
                    os.remove(model_pdb_path)

                    # if beta sheeted trajectory is detected then choose to optimise
                    if float(beta) > 15:
                        advanced_settings["soft_iterations"] = advanced_settings["soft_iterations"] + advanced_settings["optimise_beta_extra_soft"]
                        advanced_settings["temporary_iterations"] = advanced_settings["temporary_iterations"] + advanced_settings["optimise_beta_extra_temp"]
                        af_model_optimization.set_opt(num_recycles=advanced_settings["optimise_beta_recycles_design"])
                        print("Beta sheeted trajectory detected, optimising settings")

                # how many logit iterations left
                logits_iter = advanced_settings["soft_iterations"] - 50
                if logits_iter > 0:
                    print("Stage 1: Additional Logits Optimisation")
                    af_model_optimization.clear_best()
                    design_multi(af_model_optimization, template_information, advanced_settings, iters=logits_iter, e_soft=1, models=design_models, num_models=1, sample_models=advanced_settings["sample_models"], save_best=True, 
                                        dropout=False, ramp_recycles=False, binder_length=length)
                    af_model_optimization._tmp["seq_logits"] = af_model_optimization.aux["seq"]["logits"]
                    logit_plddt = get_best_plddt(af_model_optimization, length)
                    print("Optimised logit trajectory pLDDT: "+str(logit_plddt))
                else:
                    logit_plddt = initial_plddt

                # perform softmax trajectory design
                if advanced_settings["temporary_iterations"] > 0:
                    print("Stage 2: Softmax Optimisation")
                    af_model_optimization.clear_best()
                    design_multi(af_model_optimization, template_information, advanced_settings, advanced_settings["temporary_iterations"], e_temp=1e-2, models=design_models, num_models=1,
                                        sample_models=advanced_settings["sample_models"], ramp_recycles=False, save_best=True, dropout=False, soft=1, binder_length=length, decay_scale=advanced_settings["default_detargeting_decay"])
                    softmax_plddt = get_best_plddt(af_model_optimization, length)
                else:
                    softmax_plddt = logit_plddt

                # perform one hot encoding
                if softmax_plddt > 0.65:
                    print("Softmax trajectory pLDDT good, continuing: "+str(softmax_plddt))
                    if advanced_settings["hard_iterations"] > 0:
                        af_model_optimization.clear_best()
                        print("Stage 3: One-hot Optimisation")
                        design_multi(af_model_optimization, template_information, advanced_settings, advanced_settings["hard_iterations"], temp=1e-2, models=design_models, num_models=1,
                                        sample_models=advanced_settings["sample_models"], dropout=False, ramp_recycles=False, save_best=True,hard=1, soft=1, binder_length=length, decay_scale=advanced_settings["default_detargeting_decay"])
                        onehot_plddt = get_best_plddt(af_model_optimization, length)

                    if onehot_plddt > 0.65:
                        # perform greedy mutation optimisation
                        print("One-hot trajectory pLDDT good, continuing: "+str(onehot_plddt))
                        if advanced_settings["greedy_iterations"] > 0:
                            print("Stage 4: PSSM Semigreedy Optimisation")
                            af_model_optimization.clear_best()
                            design_semigreedy_multi(af_model_optimization, template_information, advanced_settings, iters=advanced_settings["greedy_iterations"], tries=greedy_tries, models=design_models, num_models=1, sample_models=advanced_settings["sample_models"], save_best=True, dropout=False, hard=1, soft=0, binder_length=length)
                            onehot_plddt = get_best_plddt(af_model_optimization, length)

                        if advanced_settings["mcmc_iterations"] > 0:
                            print("Stage 4: MCMC  Optimisation")
                            # design by using random mutations that decrease loss
                            half_life = round(advanced_settings["mcmc_iterations"] / 5, 0)
                            t_mcmc = 0.01
                            design_mcmc_multi(af_model_optimization, template_information, advanced_settings, steps=advanced_settings["mcmc_iterations"], half_life=half_life, T_init=t_mcmc, mutation_rate=1, seq_logits=None, sample_models=(advanced_settings["sample_models"]), save_best=True, verbose=True)


                    else:   
                        for template in template_information:
                            update_failures(template_information[template]["failure_csv"], 'Trajectory_one-hot_pLDDT')
                        af_model_optimization.aux['log']['terminate'] = "Skip"
                        print("One-hot trajectory pLDDT too low to continue: "+str(onehot_plddt))
                        passing=False
                        return passing, af_model_optimization

                else:
                    for template in template_information:
                        update_failures(template_information[template]["failure_csv"], 'Trajectory_softmax_pLDDT')
                    af_model_optimization.aux['log']['terminate'] = "Skip"
                    print("Softmax trajectory pLDDT too low to continue: "+str(softmax_plddt))
                    passing=False
                    return passing, af_model_optimization
            else:
                for template in template_information:
                    update_failures(template_information[template]["failure_csv"], 'Trajectory_logits_pLDDT')
                print("Initial trajectory pLDDT too low to continue: "+str(initial_plddt))
                af_model_optimization.aux['log']['terminate'] = "Skip"
                passing=False
                return passing, af_model_optimization
    else:
        print("ERROR: No valid design model selected")
        exit()

    af_model_optimization.set_seq()
    af_model_optimization.aux["log"]["terminate"] = ""
    binder_seq=af_model_optimization.get_seq(get_best=True)[0]
    trajectory_dir=design_paths["Trajectory"]

    for template in template_information:
        template_information[template]["trajectory_model"], template_information[template]["trajectory_pdb"] = process_trajectory_complex(
            binder_sequence=binder_seq, 
            design_name=design_name, 
            target_pdb=template_information[template]["template_pdb"], 
            chain=chain, 
            length=length, 
            trajectory_dir=trajectory_dir, 
            prediction_models=[0], 
            appendix=template_information[template]["template_name"], 
            failure_csv=template_information[template]["failure_csv"], 
            advanced_settings=advanced_settings
        )

    for template in template_information:
        if template_information[template]["trajectory_model"].aux["log"]["terminate"]:
            passing = False
            if template_information[template]["trajectory_model"].aux["log"]["terminate"] != "":
                shutil.move(template_information[template]["trajectory_pdb"], design_paths[f"Trajectory/{template_information[template]['trajectory_model'].aux['log']['terminate']}"])

    ### get the sampled sequence for plotting
    af_model_optimization.get_seqs()
    if advanced_settings["save_design_trajectory_plots"]:
        plot_trajectory(af_model_optimization, design_name, design_paths)

    return passing, template_information
        


def step_multi(model_to_optimize, template_information, advanced_settings, lr_scale=1.0, num_recycles=None, num_models=None, sample_models=None, models=None, backprop=True, callback=None, save_best=False, verbose=1, binder_length=None, iter_idx=None, decay_iters=None, decay_scale = None):
    # Multi-template aggregation controls, implemented as in
    # multi_colabdesign_utils_patched.py, including its defaults. Note those defaults
    # turn shuffling and RMS normalisation ON when the key is absent, so a settings
    # file that predates these keys changes behaviour; both settings files shipped
    # with this notebook set all five explicitly. Only step_multi (stages 1-3) honours
    # them -- the MCMC and semigreedy stages still aggregate the original way.
    shuffle_templates = advanced_settings.get("multi_shuffle_templates", True)
    grad_rms_norm     = advanced_settings.get("multi_grad_rms_norm", True)
    grad_rms_eps      = advanced_settings.get("multi_grad_rms_eps", 1e-8)
    dyn_beta          = advanced_settings.get("multi_dynamic_weight_beta", 0.0)
    bottleneck_alpha  = advanced_settings.get("multi_bottleneck_alpha", 0.0)

    template_order = list(template_information.keys())
    if shuffle_templates and len(template_order) > 1:
        template_order = [str(t) for t in np.random.permutation(template_order)]

    grads, base_ws = [], []
    # run optmiization for each template 
    for template in template_order:
        # empty & set template losses:
        template_information[template]["logging"] = {}

        # swap batch for current template:
        if template_information[template]["batch"] != None:
            model_to_optimize._inputs['batch'] = template_information[template]["batch"]
        # swap the hotspot too: prep_inputs() left opt["hotspot"] on whichever template was prepped last
        if template_information[template].get("hotspot") is None:
            model_to_optimize.opt.pop("hotspot", None)
        else:
            model_to_optimize.opt["hotspot"] = template_information[template]["hotspot"]

        # swap weights for targeting/detargeting:
        if template_information[template]["template_targeting_strategy"] == "target":
            model_to_optimize.opt["weights"].update(
                {"pae":(advanced_settings["weights_pae_intra"]   ),
                "plddt":(advanced_settings["weights_plddt"]      ),
                "i_pae":(advanced_settings["weights_pae_inter"]  ),
                "con":(advanced_settings["weights_con_intra"]    ),
                "i_con":(advanced_settings["weights_con_inter"]  ),
                "i_ptm":(advanced_settings["weights_iptm"]       ),
    })
        # scaling the weights according to detargeting scale
        elif template_information[template]["template_targeting_strategy"] == "detarget":
            model_to_optimize.opt["weights"].update(
                {"pae":(advanced_settings["weights_pae_intra"]   *(advanced_settings["detargeting_scaling_weights_pae_intra"])),
                "plddt":(advanced_settings["weights_plddt"]      *(advanced_settings["detargeting_scaling_weights_plddt"])),
                "i_pae":(advanced_settings["weights_pae_inter"]  *(advanced_settings["detargeting_scaling_weights_pae_inter"])),
                "con":(advanced_settings["weights_con_intra"]    *(advanced_settings["detargeting_scaling_weights_con_intra"])),
                "i_con":(advanced_settings["weights_con_inter"]  *(advanced_settings["detargeting_scaling_weights_con_inter"])),
                "i_ptm":(advanced_settings["weights_iptm"]       *(advanced_settings["detargeting_scaling_weights_iptm"])),
    })
            # Apply decay scaling if enabled
            if decay_iters is not None and iter_idx is not None:
                decay_scale = min(1.0, 0.1 + 0.9 * (iter_idx / decay_iters))
            # print("detargeting weights: ", model_to_optimize.opt["weights"])
            
        else:
            raise ValueError("No senseful design strategy has been chose, please validate this entry: ", template_information[template]["template_name"])
        
        # run one optimization step
        model_to_optimize.run(num_recycles=num_recycles, num_models=num_models, sample_models=sample_models,
             models=models, backprop=backprop, callback=callback)
        
        grad_t = copy.deepcopy(model_to_optimize.aux["grad"]['seq'])
        if template_information[template]["template_targeting_strategy"] == "detarget" and decay_scale is not None:
            grad_t = decay_scale * grad_t
        grads.append(grad_t)
        # float, not int: a fractional weight used to raise ValueError as a string
        # ("0.5") or silently truncate to 0 as a number, dropping the template
        try:
            base_ws.append(float(template_information[template]["loss_weighting"]))
        except (TypeError, ValueError):
            base_ws.append(1.0)

        # store gradient and loss
        template_information[template]["logging"]["loss_1"] = model_to_optimize.aux["log"]['loss']
        template_information[template]["logging"]["loss_2"] = model_to_optimize.aux['loss']
        
        template_information[template]["logging"]["pae_1"] = copy.deepcopy(model_to_optimize.aux["log"]['pae'])
        template_information[template]["logging"]["pae_2"] = copy.deepcopy(model_to_optimize.aux['losses']['pae'])

        template_information[template]["logging"]["i_pae_1"] = copy.deepcopy(model_to_optimize.aux["log"]['i_pae'])
        template_information[template]["logging"]["i_pae_2"] = copy.deepcopy(model_to_optimize.aux['losses']['i_pae'])

        template_information[template]["logging"]["con_1"] = copy.deepcopy(model_to_optimize.aux["log"]['con'])
        template_information[template]["logging"]["con_2"] = copy.deepcopy(model_to_optimize.aux['losses']['con'])

        template_information[template]["logging"]["i_con_1"] = copy.deepcopy(model_to_optimize.aux["log"]['i_con'])
        template_information[template]["logging"]["i_con_2"] = copy.deepcopy(model_to_optimize.aux['losses']['i_con'])

        template_information[template]["logging"]["plddt_1"] = copy.deepcopy(model_to_optimize.aux["log"]['plddt'])
        template_information[template]["logging"]["plddt_2"] = copy.deepcopy(model_to_optimize.aux['losses']['plddt'])

        template_information[template]["logging"]["ptm_1"] = copy.deepcopy(model_to_optimize.aux["log"]['ptm'])

        template_information[template]["logging"]["i_ptm_1"] = copy.deepcopy(model_to_optimize.aux["log"]['i_ptm'])
        # setting up the loss for the target plddt. 
        # template_information[template]["logging"]["plddt_target"] = copy.deepcopy(model_to_optimize.aux["plddt"][:-binder_length])

    # ----- aggregate the per-template gradients -----
    base_ws = np.asarray(base_ws, dtype=np.float32)
    losses_1 = np.asarray([template_information[t]["logging"]["loss_1"] for t in template_order],
                          dtype=np.float32)
    losses_2 = np.asarray([template_information[t]["logging"]["loss_2"] for t in template_order],
                          dtype=np.float32)

    # optional dynamic reweighting towards worst templates (uses logged loss_1)
    # dyn_beta=0 disables (weights remain base_ws). The softmax sums to 1, so enabling
    # this also scales the aggregate gradient down by roughly the template count.
    if dyn_beta is not None and float(dyn_beta) > 0.0 and len(losses_1) > 1:
        # shift improves numerical stability and makes beta scale more interpretable
        L_shift = losses_1 - np.mean(losses_1)
        dyn = softmax(float(dyn_beta) * L_shift)
        ws = base_ws * dyn
    else:
        ws = base_ws

    # avoid degenerate all-zero weights
    if not np.isfinite(ws).all() or float(np.sum(ws)) <= 0.0:
        ws = np.ones_like(base_ws, dtype=np.float32)

    running_gradient = None
    for grad_t, w in zip(grads, ws):
        if grad_rms_norm:
            # put every template on the same gradient scale, so loss_weighting rather
            # than raw gradient magnitude decides the balance between them
            grad_t = grad_t / (jnp.sqrt(jnp.mean(jnp.square(grad_t))) + grad_rms_eps)
        grad_t = grad_t * float(w)
        running_gradient = grad_t if running_gradient is None else running_gradient + grad_t

    current_sequence = copy.deepcopy(model_to_optimize.get_seq()[0])
    model_to_optimize.aux["grad"]['seq'] = running_gradient
    if model_to_optimize.opt["norm_seq_grad"]: model_to_optimize._norm_seq_grad()
    model_to_optimize._state, model_to_optimize.aux["grad"] = model_to_optimize._optimizer(model_to_optimize._state, model_to_optimize.aux["grad"], model_to_optimize._params)
    # apply gradients
    lr = model_to_optimize.opt["learning_rate"] * lr_scale
    model_to_optimize._params = jax.tree_util.tree_map(lambda x,g:x-lr*g, model_to_optimize._params, model_to_optimize.aux["grad"])
    
    # combine log/losses for tracking:
    # model_to_optimize.aux["log"]['loss'] += copy.deepcopy(np.sum([template_information[template]["logging"]["loss_1"] for template in template_information]))
    # model_to_optimize.aux['loss'] += copy.deepcopy(np.sum([template_information[template]["logging"]["loss_2"] for template in template_information]))
    # Tracking loss: weighted mean (default) or soft bottleneck (softmax over
    # per-template losses, approaching the worst template as alpha grows).
    # best_metric='loss' selects the saved design on this value.
    ws_sum = float(np.sum(ws)) if float(np.sum(ws)) > 0 else 1.0
    weighted_mean_loss1 = float(np.sum(ws * losses_1) / ws_sum)
    weighted_mean_loss2 = float(np.sum(ws * losses_2) / ws_sum)

    if bottleneck_alpha is not None and float(bottleneck_alpha) > 0.0 and len(losses_1) > 1:
        p = softmax(float(bottleneck_alpha) * (losses_1 - np.max(losses_1)))
        track_loss1 = float(np.sum(p * losses_1))
        track_loss2 = float(np.sum(p * losses_2))
    else:
        track_loss1 = weighted_mean_loss1
        track_loss2 = weighted_mean_loss2

    model_to_optimize.aux["log"]['loss'] = copy.deepcopy(track_loss1)
    model_to_optimize.aux["loss"] = copy.deepcopy(track_loss2)
    
    model_to_optimize.aux["log"]['pae'] = copy.deepcopy(np.mean([template_information[template]["logging"]["pae_1"] for template in template_information]))
    model_to_optimize.aux["losses"]['pae'] = copy.deepcopy(np.mean([template_information[template]["logging"]["pae_2"] for template in template_information]))

    model_to_optimize.aux["log"]['i_pae'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_pae_1"] for template in template_information]))
    model_to_optimize.aux["losses"]['i_pae'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_pae_2"] for template in template_information]))

    model_to_optimize.aux["log"]['con'] = copy.deepcopy(np.mean([template_information[template]["logging"]["con_1"] for template in template_information]))
    model_to_optimize.aux["losses"]['con'] = copy.deepcopy(np.mean([template_information[template]["logging"]["con_2"] for template in template_information]))

    model_to_optimize.aux["log"]['i_con'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_con_1"] for template in template_information]))
    model_to_optimize.aux["losses"]['i_con'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_con_2"] for template in template_information]))

    model_to_optimize.aux["log"]['plddt'] = copy.deepcopy(np.mean([template_information[template]["logging"]["plddt_1"] for template in template_information]))
    model_to_optimize.aux["losses"]['plddt'] = copy.deepcopy(np.mean([template_information[template]["logging"]["plddt_2"] for template in template_information]))

    model_to_optimize.aux["log"]['ptm'] = copy.deepcopy(np.mean([template_information[template]["logging"]["ptm_1"] for template in template_information]))

    model_to_optimize.aux["log"]['i_ptm'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_ptm_1"] for template in template_information]))

    # model_to_optimize.aux["log"]['sequences'] = current_sequence
    # tracking loss for the plddt of the target.
    # model_to_optimize.aux["log"]["plddt_target"] = copy.deepcopy(np.mean([template_information[template]["logging"]["plddt_target"] for template in template_information]))
    
    # save results
    model_to_optimize._save_results(save_best=save_best, verbose=verbose)
    # increment
    model_to_optimize._k += 1
   
   


def design_multi(model_to_optimize, template_information, advanced_settings, iters=100,
             soft=0.0, e_soft=None,
             temp=1.0, e_temp=None,
             hard=0.0, e_hard=None,
             step=1.0, e_step=None,
             dropout=True, opt=None, weights=None, 
             num_recycles=None, ramp_recycles=False, 
             num_models=None, sample_models=None, models=None,
             backprop=True, callback=None, save_best=False, verbose=1, binder_length=None, decay_iters=None, decay_scale=None):
    # update options/settings (if defined)

    # print(binder_length)

    model_to_optimize.set_opt(opt, dropout=dropout)
    model_to_optimize.set_weights(weights)

    m = {"soft":[soft,e_soft],"temp":[temp,e_temp],
         "hard":[hard,e_hard],"step":[step,e_step]}
    m = {k:[s,(s if e is None else e)] for k,(s,e) in m.items()}

    if ramp_recycles:
        if num_recycles is None:
            num_recycles = model_to_optimize.opt["num_recycles"]
        m["num_recycles"] = [0,num_recycles]
    
    for i in range(iters):
        for k,(s,e) in m.items():
            if k == "temp":
                model_to_optimize.set_opt({k:(e+(s-e)*(1-(i+1)/iters)**2)})
            else:
                v = (s+(e-s)*((i+1)/iters))
                if k == "step": step = v
                elif k == "num_recycles": num_recycles = round(v)
                else: model_to_optimize.set_opt({k:v})

        # decay learning rate based on temperature
        lr_scale = step * ((1 - model_to_optimize.opt["soft"]) + (model_to_optimize.opt["soft"] * model_to_optimize.opt["temp"]))   

        step_multi(model_to_optimize, template_information, advanced_settings=advanced_settings, lr_scale=lr_scale, num_recycles=num_recycles, num_models=num_models, sample_models=sample_models, models=models, backprop=backprop, callback=callback, save_best=save_best, verbose=verbose, binder_length=binder_length, iter_idx=i, decay_iters=decay_iters, decay_scale=decay_scale)


def design_mcmc_multi(model_to_optimize, template_information, advanced_settings, steps=500, half_life=200, T_init=0.01, mutation_rate=1, seq_logits=None, save_best=True, sample_models=True, verbose=1,  **kwargs):
   # gather settings
    model_flags = {k:kwargs.pop(k,None) for k in ["num_models","sample_models","models"]}

    # initialize with set numbers
    plddt, best_loss, current_loss = None, np.inf, np.inf 
    current_seq = (model_to_optimize._params["seq"] + model_to_optimize._inputs["bias"]).argmax(-1)
    if seq_logits is None: seq_logits = 0

    for template in template_information:  
        template_information[template]["MCMC_loss_value_logging"] = {0: np.inf}

    # run MCMC algorithm
    print("Running MCMC with simulated annealing...")
    for i in range(steps):

        # update temperature
        T = T_init * (np.exp(np.log(0.5) / half_life) ** i) 

        # mutate sequence
        if i == 0:
            mut_seq = current_seq
            last_accepted_index = 0
        else:
            mut_seq = model_to_optimize._mutate(
                seq=current_seq, 
                plddt=plddt,
                logits=seq_logits + model_to_optimize._inputs["bias"],
                mutation_rate=mutation_rate
            )
        # get loss for each of the templates that we have
        for template in template_information:
            
            # if template_information[template]["template_targeting_strategy"] != "target":
            #     continue
            
            # swap batch for current template:
            if template_information[template]["batch"] != None:
                model_to_optimize._inputs['batch'] = template_information[template]["batch"]
            # swap the hotspot too: prep_inputs() left opt["hotspot"] on whichever template was prepped last
            if template_information[template].get("hotspot") is None:
                model_to_optimize.opt.pop("hotspot", None)
            else:
                model_to_optimize.opt["hotspot"] = template_information[template]["hotspot"]

            model_nums = model_to_optimize._get_model_nums(**model_flags)
            aux = model_to_optimize.predict(seq=mut_seq, return_aux=True, verbose=False, model_nums=model_nums, **kwargs)
            loss = aux["log"]["loss"]

            template_information[template]["logging"]["loss_1"] = model_to_optimize.aux["log"]['loss']
            template_information[template]["logging"]["loss_2"] = model_to_optimize.aux['loss']
            
            template_information[template]["logging"]["pae_1"] = copy.deepcopy(model_to_optimize.aux["log"]['pae'])
            template_information[template]["logging"]["pae_2"] = copy.deepcopy(model_to_optimize.aux['losses']['pae'])

            template_information[template]["logging"]["i_pae_1"] = copy.deepcopy(model_to_optimize.aux["log"]['i_pae'])
            template_information[template]["logging"]["i_pae_2"] = copy.deepcopy(model_to_optimize.aux['losses']['i_pae'])

            template_information[template]["logging"]["con_1"] = copy.deepcopy(model_to_optimize.aux["log"]['con'])
            template_information[template]["logging"]["con_2"] = copy.deepcopy(model_to_optimize.aux['losses']['con'])

            template_information[template]["logging"]["i_con_1"] = copy.deepcopy(model_to_optimize.aux["log"]['i_con'])
            template_information[template]["logging"]["i_con_2"] = copy.deepcopy(model_to_optimize.aux['losses']['i_con'])

            template_information[template]["logging"]["plddt_1"] = copy.deepcopy(model_to_optimize.aux["log"]['plddt'])
            template_information[template]["logging"]["plddt_2"] = copy.deepcopy(model_to_optimize.aux['losses']['plddt'])

            template_information[template]["logging"]["ptm_1"] = copy.deepcopy(model_to_optimize.aux["log"]['ptm'])

            template_information[template]["logging"]["i_ptm_1"] = copy.deepcopy(model_to_optimize.aux["log"]['i_ptm'])


            if template_information[template]["template_targeting_strategy"] == "target":
                template_information[template]["MCMC_loss_value_logging"][i+1] = loss * int(template_information[template]["loss_weighting"])
            if template_information[template]["template_targeting_strategy"] == "detarget":
                # add a buffer for detargeting, as we want to allow for slight increase in loss 
                template_information[template]["MCMC_loss_value_logging"][i+1] = (loss - 0.1) * int(template_information[template]["loss_weighting"])

        total_loss_prev = sum(
            template_information[template]["MCMC_loss_value_logging"][last_accepted_index] 
            for template in template_information
        )
        total_loss_current = sum(
            template_information[template]["MCMC_loss_value_logging"][i+1]
            for template in template_information
        )

        loss_delta = total_loss_prev - total_loss_current

        # target_improved = (delta_target < 0)
        # if delta_detarget: detarget_tradeoff = (delta_detarget > 0 and delta_previous >= 0)
        # annealing_accept = (np.random.uniform() < np.exp(-delta_target / T))
        # # if annealing_accept: print(annealing_accept)

        # if accept_on_first or target_improved or detarget_tradeoff or annealing_accept:
        if loss_delta > 0 or i == 0:
            print("accepted MCMC:", loss)
            # accept
            last_accepted_index = i + 1
            (current_seq,current_loss) = (mut_seq,loss)
            
            plddt = aux["all"]["plddt"].mean(0)
            plddt = plddt[model_to_optimize._target_len:] if model_to_optimize.protocol == "binder" else plddt[:model_to_optimize._len]
            
            if loss < best_loss:
                (best_loss, model_to_optimize._k) = (loss, i)
                model_to_optimize.set_seq(seq=current_seq, bias=model_to_optimize._inputs["bias"])

                model_to_optimize.aux["log"]['loss'] = copy.deepcopy(np.mean([template_information[template]["logging"]["loss_1"] for template in template_information]))
                model_to_optimize.aux["loss"] = copy.deepcopy(np.mean([template_information[template]["logging"]["loss_2"] for template in template_information]))
                
                model_to_optimize.aux["log"]['pae'] = copy.deepcopy(np.mean([template_information[template]["logging"]["pae_1"] for template in template_information]))
                model_to_optimize.aux["losses"]['pae'] = copy.deepcopy(np.mean([template_information[template]["logging"]["pae_2"] for template in template_information]))

                model_to_optimize.aux["log"]['i_pae'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_pae_1"] for template in template_information]))
                model_to_optimize.aux["losses"]['i_pae'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_pae_2"] for template in template_information]))

                model_to_optimize.aux["log"]['con'] = copy.deepcopy(np.mean([template_information[template]["logging"]["con_1"] for template in template_information]))
                model_to_optimize.aux["losses"]['con'] = copy.deepcopy(np.mean([template_information[template]["logging"]["con_2"] for template in template_information]))

                model_to_optimize.aux["log"]['i_con'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_con_1"] for template in template_information]))
                model_to_optimize.aux["losses"]['i_con'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_con_2"] for template in template_information]))

                model_to_optimize.aux["log"]['plddt'] = copy.deepcopy(np.mean([template_information[template]["logging"]["plddt_1"] for template in template_information]))
                model_to_optimize.aux["losses"]['plddt'] = copy.deepcopy(np.mean([template_information[template]["logging"]["plddt_2"] for template in template_information]))

                model_to_optimize.aux["log"]['ptm'] = copy.deepcopy(np.mean([template_information[template]["logging"]["ptm_1"] for template in template_information]))

                model_to_optimize.aux["log"]['i_ptm'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_ptm_1"] for template in template_information]))


                model_to_optimize._save_results(save_best=save_best, verbose=True)

    return 0 


    
def design_semigreedy_multi(model_to_optimize, template_information, advanced_settings, iters=20, tries=8, dropout=False,
                        save_best=True, seq_logits=None, e_tries=None, binder_length=None, **kwargs):
    '''semigreedy search''' 
    if e_tries is None: e_tries = tries

    # get starting sequence
    if hasattr(model_to_optimize,"aux"):
        seq = model_to_optimize.aux["seq"]["logits"].argmax(-1)
    else:
        seq = (model_to_optimize._params["seq"] + model_to_optimize._inputs["bias"]).argmax(-1)


    # bias sampling towards the defined bias
    if seq_logits is None: seq_logits = 0
    
    model_flags = {k:kwargs.pop(k,None) for k in ["num_models","sample_models","models"]}
    verbose = kwargs.pop("verbose",1)

    # get current plddt
    aux = model_to_optimize.predict(seq, return_aux=True, verbose=False, **model_flags, **kwargs)
    plddt = model_to_optimize.aux["plddt"]
    plddt = plddt[model_to_optimize._target_len:] if model_to_optimize.protocol == "binder" else plddt[:model_to_optimize._len]

    # optimize!
    if verbose:
        print("Running semigreedy optimization...")
    
    for i in range(iters):
        buff = []
        model_nums = model_to_optimize._get_model_nums(**model_flags)
        num_tries = (tries+(e_tries-tries)*((i+1)/iters))
        for t in range(int(num_tries)):
            mut_seq = model_to_optimize._mutate(seq=seq, plddt=plddt, logits=seq_logits + model_to_optimize._inputs["bias"])

            for template in template_information:
                # empty & set template losses:
                template_information[template]["logging"] = {}

                # swap batch for current template:
                if template_information[template]["batch"] != None:
                    model_to_optimize._inputs['batch'] = template_information[template]["batch"]
                # swap the hotspot too: prep_inputs() left opt["hotspot"] on whichever template was prepped last
                if template_information[template].get("hotspot") is None:
                    model_to_optimize.opt.pop("hotspot", None)
                else:
                    model_to_optimize.opt["hotspot"] = template_information[template]["hotspot"]
                
                # swap weights for targeting/detargeting:
                if template_information[template]["template_targeting_strategy"] == "target":
                    model_to_optimize.opt["weights"].update(
                        {"pae":(advanced_settings["weights_pae_intra"]   ),
                        "plddt":(advanced_settings["weights_plddt"]      ),
                        "i_pae":(advanced_settings["weights_pae_inter"]  ),
                        "con":(advanced_settings["weights_con_intra"]    ),
                        "i_con":(advanced_settings["weights_con_inter"]  ),
                        "i_ptm":(advanced_settings["weights_iptm"]       ),
                        })
                    # print("targeting weights: ", model_to_optimize.opt["weights"])
                # scaling the weights according to detargeting scale
                elif template_information[template]["template_targeting_strategy"] == "detarget":
                    model_to_optimize.opt["weights"].update(
                        {"pae":(advanced_settings["weights_pae_intra"]   *(advanced_settings["detargeting_scaling_weights_pae_intra"])),
                        "plddt":(advanced_settings["weights_plddt"]      *(advanced_settings["detargeting_scaling_weights_plddt"])),
                        "i_pae":(advanced_settings["weights_pae_inter"]  *(advanced_settings["detargeting_scaling_weights_pae_inter"])),
                        "con":(advanced_settings["weights_con_intra"]    *(advanced_settings["detargeting_scaling_weights_con_intra"])),
                        "i_con":(advanced_settings["weights_con_inter"]  *(advanced_settings["detargeting_scaling_weights_con_inter"])),
                        "i_ptm":(advanced_settings["weights_iptm"]       *(advanced_settings["detargeting_scaling_weights_iptm"])),
            })
                    # print("detargeting weights: ", model_to_optimize.opt["weights"])
                else:
                    raise ValueError("No senseful design strategy has been chose, please validate this entry: ", template_information[template]["template_name"])
                

                # run one optimization step
                aux = model_to_optimize.predict(seq=mut_seq, return_aux=True, model_nums=model_nums, verbose=False, **kwargs)

                # store gradient and loss
                template_information[template]["logging"]["loss_1"] = model_to_optimize.aux["log"]['loss']
                template_information[template]["logging"]["loss_2"] = model_to_optimize.aux['loss']
                
                template_information[template]["logging"]["pae_1"] = copy.deepcopy(model_to_optimize.aux["log"]['pae'])
                template_information[template]["logging"]["pae_2"] = copy.deepcopy(model_to_optimize.aux['losses']['pae'])

                template_information[template]["logging"]["i_pae_1"] = copy.deepcopy(model_to_optimize.aux["log"]['i_pae'])
                template_information[template]["logging"]["i_pae_2"] = copy.deepcopy(model_to_optimize.aux['losses']['i_pae'])

                template_information[template]["logging"]["con_1"] = copy.deepcopy(model_to_optimize.aux["log"]['con'])
                template_information[template]["logging"]["con_2"] = copy.deepcopy(model_to_optimize.aux['losses']['con'])

                template_information[template]["logging"]["i_con_1"] = copy.deepcopy(model_to_optimize.aux["log"]['i_con'])
                template_information[template]["logging"]["i_con_2"] = copy.deepcopy(model_to_optimize.aux['losses']['i_con'])

                template_information[template]["logging"]["plddt_1"] = copy.deepcopy(model_to_optimize.aux["log"]['plddt'])
                template_information[template]["logging"]["plddt_2"] = copy.deepcopy(model_to_optimize.aux['losses']['plddt'])

                template_information[template]["logging"]["ptm_1"] = copy.deepcopy(model_to_optimize.aux["log"]['ptm'])

                template_information[template]["logging"]["i_ptm_1"] = copy.deepcopy(model_to_optimize.aux["log"]['i_ptm'])
                # setting up the loss for the target plddt. 
                # template_information[template]["logging"]["plddt_target"] = copy.deepcopy(model_to_optimize.aux["plddt"][:-binder_length])
            
            buff.append({"aux":aux, "seq":np.array(mut_seq)})

            # combine log/losses for tracking:
            # model_to_optimize.aux["log"]['loss'] += copy.deepcopy(np.sum([template_information[template]["logging"]["loss_1"] for template in template_information]))
            # model_to_optimize.aux['loss'] += copy.deepcopy(np.sum([template_information[template]["logging"]["loss_2"] for template in template_information]))

            model_to_optimize.aux["log"]['loss'] = copy.deepcopy(np.mean([template_information[template]["logging"]["loss_1"] for template in template_information]))
            model_to_optimize.aux['loss'] = copy.deepcopy(np.mean([template_information[template]["logging"]["loss_2"] for template in template_information]))

            model_to_optimize.aux["log"]['pae'] = copy.deepcopy(np.mean([template_information[template]["logging"]["pae_1"] for template in template_information]))
            model_to_optimize.aux["losses"]['pae'] = copy.deepcopy(np.mean([template_information[template]["logging"]["pae_2"] for template in template_information]))

            model_to_optimize.aux["log"]['i_pae'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_pae_1"] for template in template_information]))
            model_to_optimize.aux["losses"]['i_pae'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_pae_2"] for template in template_information]))

            model_to_optimize.aux["log"]['con'] = copy.deepcopy(np.mean([template_information[template]["logging"]["con_1"] for template in template_information]))
            model_to_optimize.aux["losses"]['con'] = copy.deepcopy(np.mean([template_information[template]["logging"]["con_2"] for template in template_information]))

            model_to_optimize.aux["log"]['i_con'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_con_1"] for template in template_information]))
            model_to_optimize.aux["losses"]['i_con'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_con_2"] for template in template_information]))

            model_to_optimize.aux["log"]['plddt'] = copy.deepcopy(np.mean([template_information[template]["logging"]["plddt_1"] for template in template_information]))
            model_to_optimize.aux["losses"]['plddt'] = copy.deepcopy(np.mean([template_information[template]["logging"]["plddt_2"] for template in template_information]))

            model_to_optimize.aux["log"]['ptm'] = copy.deepcopy(np.mean([template_information[template]["logging"]["ptm_1"] for template in template_information]))

            model_to_optimize.aux["log"]['i_ptm'] = copy.deepcopy(np.mean([template_information[template]["logging"]["i_ptm_1"] for template in template_information]))
            
            # tracking loss for the plddt of the target. 
            # model_to_optimize.aux["log"]["plddt_target"] = copy.deepcopy(np.mean([template_information[template]["logging"]["plddt_target"] for template in template_information]))


        # accept best
        losses = [x["aux"]["loss"] for x in buff]
        best = buff[np.argmin(losses)]
        model_to_optimize.aux, seq = best["aux"], jnp.array(best["seq"])
        model_to_optimize.set_seq(seq=seq, bias=model_to_optimize._inputs["bias"])
        model_to_optimize._save_results(save_best=save_best, verbose=verbose)
        # update plddt
        plddt = best["aux"]["plddt"]
        plddt = plddt[model_to_optimize._target_len:] if model_to_optimize.protocol == "binder" else plddt[:model_to_optimize._len]
        model_to_optimize._k += 1
    

def process_trajectory_complex(
    binder_sequence,
    design_name,         # Replaces mpnn_design_name
    target_pdb,          # Path to the target PDB file
    chain,               # Chain identifier
    length,              # Binder length
    trajectory_dir,      # Directory where trajectory PDBs will be saved
    prediction_models,
    failure_csv,         # Non-default argument comes before default arguments
    appendix="",   
    advanced_settings={} # Default argument placed at the end
):
    """
    Predicts the trajectory complex using a binder sequence and relaxes the generated models.
    
    Parameters:
      prediction_model: Base prediction model (used for metrics reference)
      binder_sequence: Amino acid sequence for the binder (will be cleaned)
      design_name: Name/identifier for the design (used in file naming)
      target_pdb: File path to the target PDB structure
      chain: Chain identifier for the target
      length: Length of the binder sequence
      trajectory_dir: Directory path to save trajectory PDB files
      relaxed_dir: Directory path to save relaxed PDB files
      prediction_models: Iterable of model numbers to predict/relax
      failure_csv: Path to the failure CSV
      appendix: Additional string to append to filenames
      advanced_settings: Dictionary containing advanced settings, e.g.:
           - "num_recycles_validation": int
           - "af_params_dir": str (path to AF parameters)
           - "rm_template_seq_predict": bool
           - "rm_template_sc_predict": bool
           - "multimer_validation": bool
      seed: Optional random seed for reproducibility
      
    Returns:
      prediction_stats: Dictionary containing prediction metrics for each model.
    """
    # Initialize the trajectory prediction model using advanced settings.
    trajectory_prediction_model = mk_afdesign_model(
        protocol="binder",
        num_recycles=advanced_settings.get("num_recycles_design", 1),
        data_dir=advanced_settings.get("af_params_dir", ""),
        use_multimer=advanced_settings.get("use_multimer_design", True)
    )
    
    # Prepare the inputs using the target PDB and chain.
    trajectory_prediction_model.prep_inputs(
        pdb_filename=target_pdb,
        chain=chain,
        binder_len=length,
        rm_target_seq=advanced_settings.get("rm_template_seq_design", False),
        rm_target_sc=advanced_settings.get("rm_template_sc_design", False)
    )
    
    prediction_stats = {}
    
    # Clean binder sequence: keep only uppercase letters.
    binder_sequence = re.sub("[^A-Z]", "", binder_sequence.upper())
    
    # Loop over each model number in the prediction_models list.
    for model_num in prediction_models:
        # Construct the file path for the trajectory PDB.
        trajectory_pdb = os.path.join(trajectory_dir, f"{design_name}{appendix}.pdb")
        
        # Predict the model only if the file does not already exist.
        if not os.path.exists(trajectory_pdb):
            trajectory_prediction_model.predict(
                seq=binder_sequence,
                models=[model_num],
                num_recycles=advanced_settings.get("num_recycles_design", 1),
                verbose=False
            )
            # Save the predicted structure.
            trajectory_prediction_model.save_pdb(trajectory_pdb)
            
            # Copy the prediction metrics from the model's auxiliary log.
            prediction_metrics = trajectory_prediction_model.aux["log"].copy()  # Expected keys: plddt, ptm, i_ptm, pae, i_pae
            
            # Store rounded statistics.
            stats = {
                'pLDDT': round(prediction_metrics.get('plddt', 0), 2), 
                'pTM': round(prediction_metrics.get('ptm', 0), 2), 
                'i_pTM': round(prediction_metrics.get('i_ptm', 0), 2), 
                'pAE': round(prediction_metrics.get('pae', 0), 2), 
                'i_pAE': round(prediction_metrics.get('i_pae', 0), 2),
                # 'plddt_target': round(prediction_metrics.get('plddt_target', 0), 2)
            }
            prediction_stats[model_num+1] = stats
            print(f"Model {model_num+1} stats:", stats)
        else:
            print(f"Trajectory file {trajectory_pdb} already exists. Skipping prediction.")
        
        ca_clashes = calculate_clash_score(trajectory_pdb, 2.5, only_ca=True)
        if ca_clashes > 0:
            trajectory_prediction_model.aux['log']['terminate'] = "Clashing"
            update_failures(failure_csv, 'Trajectory_Clashes')
            print("Severe clashes detected, skipping analysis and MPNN optimisation")
            print("")
        else:
            # check if low quality prediction
            if stats['pLDDT'] < 0.7:
                trajectory_prediction_model.aux['log']['terminate'] = "LowConfidence"
                update_failures(failure_csv, 'Trajectory_final_pLDDT')
                print("Trajectory starting confidence low, skipping analysis and MPNN optimisation")
                print("")
            else:
                # does it have enough contacts to consider?
                binder_contacts = hotspot_residues(trajectory_pdb)
                binder_contacts_n = len(binder_contacts.items())

                # if less than 7 contacts then protein is floating above and is not binder
                if binder_contacts_n < 7:
                    trajectory_prediction_model.aux['log']['terminate'] = "LowConfidence"
                    update_failures(failure_csv, 'Trajectory_Contacts')
                    print("Too few contacts at the interface, skipping analysis and MPNN optimisation")
                    print("")
                else:
                    # phew, trajectory is okay! We can continue
                    trajectory_prediction_model.aux["log"]["terminate"] = ""
                    print(f"Trajectory against {appendix} successful, final pLDDT: "+str(stats['pLDDT']))    
    
    return trajectory_prediction_model, trajectory_pdb

# Define radius of gyration loss for colabdesign
def add_rg_loss(self, weight=0.1):
    '''add radius of gyration loss'''
    def loss_fn(inputs, outputs):
        xyz = outputs["structure_module"]
        ca = xyz["final_atom_positions"][:,residue_constants.atom_order["CA"]]
        ca = ca[-self._binder_len:]
        rg = jnp.sqrt(jnp.square(ca - ca.mean(0)).sum(-1).mean() + 1e-8)
        rg_th = 2.38 * ca.shape[0] ** 0.365

        rg = jax.nn.elu(rg - rg_th)
        return {"rg":rg}

    self._callbacks["model"]["loss"].append(loss_fn)
    self.opt["weights"]["rg"] = weight
# Define interface pTM loss for colabdesign
def add_i_ptm_loss(self, weight=0.1):
    def loss_iptm(inputs, outputs):
        p = 1 - get_ptm(inputs, outputs, interface=True)
        i_ptm = mask_loss(p)
        return {"i_ptm": i_ptm}
    
    self._callbacks["model"]["loss"].append(loss_iptm)
    self.opt["weights"]["i_ptm"] = weight

# add target plddt loss
def add_target_plddt_loss(self, weight: float = 0.0) -> None:
    """Optimise predicted confidence over target residues."""
    def _per_residue_plddt(outputs) -> jnp.ndarray:
        logits = outputs["predicted_lddt"]["logits"]
        num_bins = logits.shape[-1]
        centres = jnp.arange(0.5 / num_bins, 1.0, 1.0 / num_bins)
        probs = jax.nn.softmax(logits, axis=-1)
        return jnp.sum(probs * centres, axis=-1)

    def loss_fn(inputs, outputs):
        plddt = _per_residue_plddt(outputs)
        target_plddt = plddt[:self._target_len]

        mean_val = target_plddt.mean()
        loss_val = 1.0 - mean_val
        return {"plddt_target": loss_val}

    self._callbacks["model"]["loss"].append(loss_fn)
    self.opt["weights"]["plddt_target"] = weight

# add helicity loss
def add_helix_loss(self, weight=0):
    def binder_helicity(inputs, outputs):  
      if "offset" in inputs:
        offset = inputs["offset"]
      else:
        idx = inputs["residue_index"].flatten()
        offset = idx[:,None] - idx[None,:]

      # define distogram
      dgram = outputs["distogram"]["logits"]
      dgram_bins = get_dgram_bins(outputs)
      mask_2d = np.outer(np.append(np.zeros(self._target_len), np.ones(self._binder_len)), np.append(np.zeros(self._target_len), np.ones(self._binder_len)))

      x = _get_con_loss(dgram, dgram_bins, cutoff=6.0, binary=True)
      if offset is None:
        if mask_2d is None:
          helix_loss = jnp.diagonal(x,3).mean()
        else:
          helix_loss = jnp.diagonal(x * mask_2d,3).sum() + (jnp.diagonal(mask_2d,3).sum() + 1e-8)
      else:
        mask = offset == 3
        if mask_2d is not None:
          mask = jnp.where(mask_2d,mask,0)
        helix_loss = jnp.where(mask,x,0.0).sum() / (mask.sum() + 1e-8)

      return {"helix":helix_loss}
    self._callbacks["model"]["loss"].append(binder_helicity)
    self.opt["weights"]["helix"] = weight

# Get pLDDT of best model
def get_best_plddt(af_model, length):
    return round(np.mean(af_model._tmp["best"]["aux"]["plddt"][-length:]),2)


# run MPNN to generate sequences for binders
def mpnn_gen_sequence(trajectory_pdb, binder_chain, trajectory_interface_residues, advanced_settings):
    # clear GPU memory
    clear_mem()

    # initialise MPNN model
    mpnn_model = mk_mpnn_model(backbone_noise=advanced_settings["backbone_noise"], model_name=advanced_settings["model_path"], weights=advanced_settings["mpnn_weights"])

    # check whether keep the interface generated by the trajectory or whether to redesign with MPNN
    design_chains = 'A,' + binder_chain

    if advanced_settings["mpnn_fix_interface"]:
        fixed_positions = 'A,' + trajectory_interface_residues
        print("Fixing interface residues on Binder: "+trajectory_interface_residues)
    else:
        fixed_positions = 'A'

    # prepare inputs for MPNN
    mpnn_model.prep_inputs(pdb_filename=trajectory_pdb, chain=design_chains, fix_pos=fixed_positions, rm_aa=advanced_settings["omit_AAs"])

    # sample MPNN sequences in parallel
    mpnn_sequences = mpnn_model.sample_parallel(temperature=advanced_settings["sampling_temp"], num=advanced_settings["num_seqs"], batch=advanced_settings["sample_seq_parallel"])

    return mpnn_sequences

# run prediction for binder with masked template target
def masked_binder_predict_multi(trajectory_dict, binder_sequence, mpnn_design_name, prediction_models, advanced_settings, filters, design_paths):
    prediction_stats = {}

    # clean sequence
    binder_sequence = re.sub("[^A-Z]", "", binder_sequence.upper())

    # reset filtering conditionals
    pass_af2_filters = True
    filter_failures = {}

    # start prediction per AF2 model, 2 are used by default due to masked templates
    for model_num in prediction_models:
        # check to make sure prediction does not exist already
        complex_pdb = os.path.join(design_paths["MPNN"], f"{mpnn_design_name}_model{model_num+1}{trajectory_dict['template_name']}.pdb")
        if not os.path.exists(complex_pdb):
            # predict model
            trajectory_dict["complex_prediction_model"].predict(seq=binder_sequence, models=[model_num], num_recycles=advanced_settings["num_recycles_validation"], verbose=False)
            trajectory_dict["complex_prediction_model"].save_pdb(complex_pdb)
            prediction_metrics = copy.deepcopy(trajectory_dict["complex_prediction_model"].aux["log"]) # contains plddt, ptm, i_ptm, pae, i_pae

            # extract the statistics for the model
            stats = {
                'pLDDT': round(prediction_metrics['plddt'], 2), 
                'pTM': round(prediction_metrics['ptm'], 2), 
                'i_pTM': round(prediction_metrics['i_ptm'], 2), 
                'pAE': round(prediction_metrics['pae'], 2), 
                'i_pAE': round(prediction_metrics['i_pae'], 2)
            }
            prediction_stats[model_num+1] = stats

            ## get filter conditions from the file

            # List of filter conditions and corresponding keys
            filter_conditions = [
                (f"{model_num+1}_pLDDT", 'plddt', '>='),
                (f"{model_num+1}_pTM", 'ptm', '>='),
                (f"{model_num+1}_i_pTM", 'i_ptm', '>='),
                (f"{model_num+1}_pAE", 'pae', '<='),
                (f"{model_num+1}_i_pAE", 'i_pae', '<='),
            ]

            # perform initial AF2 values filtering to determine whether to skip relaxation and interface scoring
            for filter_name, metric_key, _ in filter_conditions:
                threshold = filters.get(filter_name, {}).get("threshold")
                comparison = ">=" if filters.get(filter_name, {}).get("higher") else "<="
                if threshold is not None:
                    if comparison == '>=' and prediction_metrics[metric_key] < threshold:
                        pass_af2_filters = False
                        filter_failures[filter_name] = filter_failures.get(filter_name, 0) + 1
                    elif comparison == '<=' and prediction_metrics[metric_key] > threshold:
                        pass_af2_filters = False
                        filter_failures[filter_name] = filter_failures.get(filter_name, 0) + 1

            if not pass_af2_filters:
                break

    # Update the CSV file with the failure counts
    if filter_failures:
        update_failures(trajectory_dict["failure_csv"], filter_failures)

    # AF2 filters passed, contuing with relaxation
    for model_num in prediction_models:
        complex_pdb = os.path.join(design_paths["MPNN"], f"{mpnn_design_name}_model{model_num+1}{trajectory_dict['template_name']}.pdb")
        if pass_af2_filters:
            mpnn_relaxed = os.path.join(design_paths["MPNN/Relaxed"], f"{mpnn_design_name}_model{model_num+1}{trajectory_dict['template_name']}.pdb")
            pr_relax(complex_pdb, mpnn_relaxed)
        else:
            if os.path.exists(complex_pdb):
                os.remove(complex_pdb)

    return prediction_stats, pass_af2_filters


# run prediction for binder with masked template target
def masked_binder_predict_multi_adjusted(trajectory_dict, binder_sequence, mpnn_design_name, prediction_models, advanced_settings, filters, design_paths, target_statistics):
    prediction_stats = {}
    targeting_strategy = trajectory_dict["template_targeting_strategy"]

    # clean sequence
    binder_sequence = re.sub("[^A-Z]", "", binder_sequence.upper())

    # reset filtering conditionals
    pass_af2_filters = True
    filter_failures = {}

    # start prediction per AF2 model, 2 are used by default due to masked templates
    for model_num in prediction_models:
        # check to make sure prediction does not exist already
        complex_pdb = os.path.join(design_paths["MPNN"], f"{mpnn_design_name}_model{model_num+1}{trajectory_dict['template_name']}.pdb")
        if not os.path.exists(complex_pdb):
            # predict model
            trajectory_dict["complex_prediction_model"].predict(seq=binder_sequence, models=[model_num], num_recycles=advanced_settings["num_recycles_validation"], verbose=False)
            trajectory_dict["complex_prediction_model"].save_pdb(complex_pdb)
            prediction_metrics = copy.deepcopy(trajectory_dict["complex_prediction_model"].aux["log"]) # contains plddt, ptm, i_ptm, pae, i_pae

            # extract the statistics for the model
            stats = {
                'pLDDT': round(prediction_metrics['plddt'], 2), 
                'pTM': round(prediction_metrics['ptm'], 2), 
                'i_pTM': round(prediction_metrics['i_ptm'], 2), 
                'pAE': round(prediction_metrics['pae'], 2), 
                'i_pAE': round(prediction_metrics['i_pae'], 2)
            }
            prediction_stats[model_num+1] = stats

            ## get filter conditions from the file

            # List of filter conditions and corresponding keys
            filter_conditions = [
                (f"{model_num+1}_pLDDT", 'plddt', '>='),
                (f"{model_num+1}_pTM", 'ptm', '>='),
                (f"{model_num+1}_i_pTM", 'i_ptm', '>='),
                (f"{model_num+1}_pAE", 'pae', '<='),
                (f"{model_num+1}_i_pAE", 'i_pae', '<='),
            ]

            if targeting_strategy != "detarget":
                # perform initial AF2 values filtering to determine whether to skip relaxation and interface scoring
                for filter_name, metric_key, _ in filter_conditions:
                    threshold = filters.get(filter_name, {}).get("threshold")
                    comparison = ">=" if filters.get(filter_name, {}).get("higher") else "<="
                    if threshold is not None:
                        if comparison == '>=' and prediction_metrics[metric_key] < threshold:
                            pass_af2_filters = False
                            filter_failures[filter_name] = filter_failures.get(filter_name, 0) + 1
                        elif comparison == '<=' and prediction_metrics[metric_key] > threshold:
                            pass_af2_filters = False
                            filter_failures[filter_name] = filter_failures.get(filter_name, 0) + 1

            else: 
                for filter_name, metric_key, _ in filter_conditions:
                    if "i_pTM" in filter_name:
                        threshold = filters.get(filter_name, {}).get("threshold")
                        threshold = target_statistics["i_pTM"] - threshold
                        comparison = ">=" if filters.get(filter_name, {}).get("higher") else "<="
                        if threshold is not None:
                            if comparison == '>=' and prediction_metrics[metric_key] < threshold:
                                pass_af2_filters = False
                                filter_failures[filter_name] = filter_failures.get(filter_name, 0) + 1
                            elif comparison == '<=' and prediction_metrics[metric_key] > threshold:
                                pass_af2_filters = False
                                filter_failures[filter_name] = filter_failures.get(filter_name, 0) + 1
                    
                    else: 
                        threshold = filters.get(filter_name, {}).get("threshold")
                        comparison = ">=" if filters.get(filter_name, {}).get("higher") else "<="
                        if threshold is not None:
                            if comparison == '>=' and prediction_metrics[metric_key] < threshold:
                                pass_af2_filters = False
                                filter_failures[filter_name] = filter_failures.get(filter_name, 0) + 1
                            elif comparison == '<=' and prediction_metrics[metric_key] > threshold:
                                pass_af2_filters = False
                                filter_failures[filter_name] = filter_failures.get(filter_name, 0) + 1


            if not pass_af2_filters:
                break

    # Update the CSV file with the failure counts
    if filter_failures:
        update_failures(trajectory_dict["failure_csv"], filter_failures)

    # AF2 filters passed, contuing with relaxation
    for model_num in prediction_models:
        complex_pdb = os.path.join(design_paths["MPNN"], f"{mpnn_design_name}_model{model_num+1}{trajectory_dict['template_name']}.pdb")
        if pass_af2_filters:
            mpnn_relaxed = os.path.join(design_paths["MPNN/Relaxed"], f"{mpnn_design_name}_model{model_num+1}{trajectory_dict['template_name']}.pdb")
            pr_relax(complex_pdb, mpnn_relaxed)
        else:
            if os.path.exists(complex_pdb):
                os.remove(complex_pdb)

    return prediction_stats, pass_af2_filters

# run prediction for binder alone
def predict_binder_alone(prediction_model, binder_sequence, mpnn_design_name, length, trajectory_pdb, binder_chain, prediction_models, advanced_settings, design_paths, seed=None):
    binder_stats = {}

    # prepare sequence for prediction
    binder_sequence = re.sub("[^A-Z]", "", binder_sequence.upper())
    prediction_model.set_seq(binder_sequence)

    # predict each model separately
    for model_num in prediction_models:
        # check to make sure prediction does not exist already
        binder_alone_pdb = os.path.join(design_paths["MPNN/Binder"], f"{mpnn_design_name}_model{model_num+1}.pdb")
        if not os.path.exists(binder_alone_pdb):
            # predict model
            prediction_model.predict(models=[model_num], num_recycles=advanced_settings["num_recycles_validation"], verbose=False)
            prediction_model.save_pdb(binder_alone_pdb)
            prediction_metrics = copy_dict(prediction_model.aux["log"]) # contains plddt, ptm, pae

            # align binder model to trajectory binder
            align_pdbs(trajectory_pdb, binder_alone_pdb, binder_chain, "A")

            # extract the statistics for the model
            stats = {
                'pLDDT': round(prediction_metrics['plddt'], 2), 
                'pTM': round(prediction_metrics['ptm'], 2), 
                'pAE': round(prediction_metrics['pae'], 2)
            }
            binder_stats[model_num+1] = stats

    return binder_stats



# plot design trajectory losses
def plot_trajectory(af_model, design_name, design_paths):
    metrics_to_plot = ['loss', 'plddt', 'ptm', 'i_ptm', 'con', 'i_con', 'pae', 'i_pae', 'rg', 'mpnn', 'i_ptm_spread']
    colors = ['b', 'g', 'r', 'c', 'm', 'y', 'k']

    metrics_dict = {}

    for index, metric in enumerate(metrics_to_plot):
        if metric in af_model.aux["log"]:
            # Create a new figure for each metric
            plt.figure()

            loss = af_model.get_loss(metric)
            metrics_dict[metric] = loss.tolist() if hasattr(loss, 'tolist') else list(loss)
            # Create an x axis for iterations
            iterations = range(1, len(loss) + 1)

            plt.plot(iterations, loss, label=f'{metric}', color=colors[index % len(colors)])

            # Add labels and a legend
            plt.xlabel('Iterations')
            plt.ylabel(metric)
            plt.title(design_name)
            plt.legend()
            plt.grid(True)

            # Save the plot
            plt.savefig(os.path.join(design_paths["Trajectory/Plots"], design_name+"_"+metric+".png"), dpi=150)
            
            # Close the figure
            plt.close()

    metrics_file_path = os.path.join(design_paths["Trajectory/Plots"], f"{design_name}_metrics.json")
    with open(metrics_file_path, 'w') as f:
        json.dump(metrics_dict, f, indent=4)

In [ ]:
# @title Settings and per-template bookkeeping
import sys

target_settings = TARGET
setting_file  = Path(TARGET_JSON).stem if TARGET_JSON else target_settings["general_information"]["binder_name"]
advanced_file = Path(ADVANCED_JSON).stem

design_paths = generate_directories(target_settings["general_information"]["design_path"])
trajectory_labels, design_labels, final_labels = generate_dataframe_labels()

general_information, template_information = collect_target_information(
    bindcraft_path=str(PROJECT_ROOT),
    target_settings=target_settings,
    design_labels=design_labels,
    final_labels=final_labels,
    trajectory_labels=trajectory_labels,
    args=None,
)

print(f"design_path  {general_information['design_path']}")
print(f"templates    {len(template_information)}: "
      + ", ".join(template_information[t]["template_name"] for t in template_information))
print(f"lengths      {general_information['lengths']}   target {general_information['number_of_final_designs']} designs")

## 7b. Multi-template aggregation knobs

Section 5b defines these and section 6 loaded them back. This cell overrides them
after the fact, so a sweep can change one value per run without re-editing the
`ADVANCED` dict, and records that it did.

Only `step_multi()` reads them, i.e. **stages 1-3 only**. The MCMC and semigreedy
passes of stage 4 aggregate the templates the original way regardless.

| key | what it does |
|---|---|
| `multi_grad_rms_norm` | divide each template's gradient by its own RMS before weighting, so `loss_weighting` rather than raw gradient magnitude decides the balance between templates |
| `multi_grad_rms_eps` | epsilon added to that RMS |
| `multi_shuffle_templates` | randomise the order templates are visited each step |
| `multi_dynamic_weight_beta` | `softmax(beta * (L - mean(L)))` multiplied into the base weights, tilting the gradient towards whichever template is currently worst. `0` disables |
| `multi_bottleneck_alpha` | softmax over the per-template losses for the *reported/tracked* loss, approaching the worst template as alpha grows. `0` disables |
| `multi_stage1_warmup_iters` | extra raw-logits iterations (`e_soft=0`, no softmax) before stage 1's soft ramp |



In [ ]:
# @title Multi-template aggregation knobs — override the settings JSON here
# KEEP = use whatever the JSON loaded. Any other value overrides it for this run.
KEEP = ...

MULTI_OVERRIDES = {
    "multi_grad_rms_norm":         KEEP,   # bool
    "multi_grad_rms_eps":          KEEP,   # float
    "multi_shuffle_templates":     KEEP,   # bool
    "multi_dynamic_weight_beta":   KEEP,   # float >= 0, 0 disables
    "multi_bottleneck_alpha":      KEEP,   # float >= 0, 0 disables
    "multi_stage1_warmup_iters":   KEEP,   # int
    "use_axis_alignment_loss":     KEEP,   # bool  (unvalidated, see engine docstring)
    "weights_axis_alignment_loss": KEEP,   # float
}

_applied = {}
for _k, _v in MULTI_OVERRIDES.items():
    if _v is KEEP:
        continue
    _old = advanced_settings.get(_k, "<absent>")
    if _old != _v:
        _applied[_k] = (_old, _v)
    advanced_settings[_k] = _v

print(f"aggregation knobs in effect  (base: {Path(ADVANCED_JSON).name})")
for _k in MULTI_OVERRIDES:
    _note = f"   <- was {_applied[_k][0]!r}" if _k in _applied else ""
    print(f"  {_k:30s} = {advanced_settings.get(_k, '<absent>')!r}{_note}")

# every trajectory/mpnn CSV row records which settings file it came from; that
# stamp would be a lie once the notebook has changed the values
if _applied:
    advanced_file = f"{Path(ADVANCED_JSON).stem}+nb_override"
    print(f"\n{len(_applied)} override(s) applied — rows stamped {advanced_file!r}")

_beta = float(advanced_settings.get("multi_dynamic_weight_beta", 0.0) or 0.0)
if _beta > 0.0:
    _n = len(template_information)
    print(f"\nnote: beta={_beta} makes the weights a softmax summing to 1 rather than to "
          f"{_n},\n      so the aggregate gradient is ~{_n}x smaller on top of the tilt, and "
          f"the\n      tracked loss becomes that same loss-dependent weighted mean.")

## 8. Initialise PyRosetta

In [ ]:
# @title Initialise PyRosetta
pr.init(f'-ignore_unrecognized_res -ignore_zero_occupancy -mute all '
        f'-holes:dalphaball {advanced_settings["dalphaball_path"]} '
        f'-corrections::beta_nov16 true -relax:default_repeats 1')

## 9. Run BindCraft!



In [ ]:
# ---------------------------------------------------------------------------
# Run BindCraft (multi-template)
# Port of multi_binder_design.py:main(), laid out like upstream BindCraft.ipynb.
# Deviations from the script are marked  # DEVIATION:  and listed in the
# markdown cell above. Interrupt the kernel to stop early; state is on disk,
# so re-running the cell resumes.
# ---------------------------------------------------------------------------
script_start_time = time.time()
trajectory_n = 1
accepted_designs = 0
rejected_designs = 0

# [RUN-LIMIT] Cap this run at a small number of trajectories (quick test).
# Set to None to restore the normal campaign, which otherwise runs until
# number_of_final_designs accepted binders or advanced_settings['max_trajectories'].
MAX_TRAJECTORIES_THIS_RUN = 3

while True:
    # [RUN-LIMIT] hard cap on trajectories attempted this run (counts every
    # attempt, including ones that terminate early, so it is exactly one).
    if MAX_TRAJECTORIES_THIS_RUN is not None and trajectory_n > MAX_TRAJECTORIES_THIS_RUN:
        print(f"[RUN-LIMIT] Reached {MAX_TRAJECTORIES_THIS_RUN} trajectory(ies); stopping.")
        break

    # enough designs passing every template's filters?
    if check_accepted_designs_multi(design_paths, template_information, general_information,
                                    final_labels, advanced_settings, target_settings, design_labels):
        break

    if check_n_trajectories(design_paths, advanced_settings):
        break

    trajectory_start_time = time.time()
    seed = int(np.random.randint(0, high=999999, size=1, dtype=int)[0])
    samples = np.arange(min(general_information["lengths"]), max(general_information["lengths"]) + 1)
    length = int(np.random.choice(samples))
    helicity_value = load_helicity(advanced_settings)

    design_name = f'{general_information["binder_name"]}_l{length}_s{seed}'
    trajectory_dirs = ["Trajectory", "Trajectory/Relaxed", "Trajectory/LowConfidence", "Trajectory/Clashing"]
    if any(os.path.exists(os.path.join(design_paths[d], design_name + ".pdb")) for d in trajectory_dirs):
        trajectory_n += 1
        continue

    print(f"\n=== Trajectory {trajectory_n}: {design_name} "
          f"(accepted so far: {accepted_designs}) ===")

    ### Stage 1-4: hallucinate one binder against every template at once
    passing, templates_run = binder_hallucination_multi(
        template_information=copy.deepcopy(template_information),
        design_paths=design_paths,
        design_name=design_name,
        advanced_settings=advanced_settings,
        length=length,
        seed=seed,
        chain=general_information["chains"],
        helicity_value=helicity_value,
        design_models=design_models,
    )

    trajectory_time = time.time() - trajectory_start_time
    trajectory_time_text = "%d hours, %d minutes, %d seconds" % (
        int(trajectory_time // 3600), int((trajectory_time % 3600) // 60), int(trajectory_time % 60))
    print(f"Trajectory took: {trajectory_time_text}")

    if not passing:
        trajectory_n += 1
        continue

    # a template that terminated (clashes / low confidence) disqualifies the trajectory
    terminated = False
    for template in templates_run:
        metrics = copy_dict(templates_run[template]["trajectory_model"].aux["log"])
        templates_run[template]["trajecotry_metrics"] = {
            k: round(v, 2) if isinstance(v, float) else v for k, v in metrics.items()}
        if templates_run[template]["trajecotry_metrics"]["terminate"] != "":
            terminated = True
    if terminated:
        trajectory_n += 1
        continue

    ### Relax and score the trajectory against each template
    interface_residue_sets = []
    for template in templates_run:
        tpl = templates_run[template]
        tpl["trajectory_pdb_relaxed"] = os.path.join(
            design_paths["Trajectory/Relaxed"], design_name + tpl["template_name"] + ".pdb")
        pr_relax(tpl["trajectory_pdb"], tpl["trajectory_pdb_relaxed"])

        binder_chain = general_information["binder_chain"]

        tpl["num_clashes_trajectory"] = calculate_clash_score(tpl["trajectory_pdb"])
        tpl["num_clashes_relaxed"] = calculate_clash_score(tpl["trajectory_pdb_relaxed"])

        (tpl["trajectory_alpha"], tpl["trajectory_beta"], tpl["trajectory_loops"],
         tpl["trajectory_alpha_interface"], tpl["trajectory_beta_interface"],
         tpl["trajectory_loops_interface"], tpl["trajectory_i_plddt"],
         tpl["trajectory_ss_plddt"]) = calc_ss_percentage(
            tpl["trajectory_pdb"], advanced_settings, binder_chain)

        (tpl["trajectory_interface_scores"], tpl["trajectory_interface_AA"],
         tpl["trajectory_interface_residues"]) = score_interface(
            tpl["trajectory_pdb_relaxed"], binder_chain)
        interface_residue_sets.append(set(tpl["trajectory_interface_residues"].split(',')))

        trajectory_sequence = tpl["trajectory_model"].get_seq(get_best=True)[0]
        traj_seq_notes = validate_design_sequence(
            trajectory_sequence, tpl["num_clashes_relaxed"], advanced_settings)
        trajectory_target_rmsd = target_pdb_rmsd(
            tpl["trajectory_pdb"], tpl["template_pdb"], general_information["chains"])

        scores = tpl["trajectory_interface_scores"]
        insert_data(tpl["trajectory_csv"], [
            design_name, advanced_settings["design_algorithm"], length, seed, helicity_value,
            tpl["template_hostspot_residues"], trajectory_sequence, tpl["trajectory_interface_residues"],
            tpl["trajecotry_metrics"]['plddt'], tpl["trajecotry_metrics"]['ptm'],
            tpl["trajecotry_metrics"]['i_ptm'], tpl["trajecotry_metrics"]['pae'],
            tpl["trajecotry_metrics"]['i_pae'], tpl["trajectory_i_plddt"], tpl["trajectory_ss_plddt"],
            tpl["num_clashes_trajectory"], tpl["num_clashes_relaxed"],
            scores['binder_score'], scores['surface_hydrophobicity'], scores['interface_sc'],
            scores['interface_packstat'], scores['interface_dG'], scores['interface_dSASA'],
            scores['interface_dG_SASA_ratio'], scores['interface_fraction'],
            scores['interface_hydrophobicity'], scores['interface_nres'],
            scores['interface_interface_hbonds'], scores['interface_hbond_percentage'],
            scores['interface_delta_unsat_hbonds'], scores['interface_delta_unsat_hbonds_percentage'],
            tpl["trajectory_alpha_interface"], tpl["trajectory_beta_interface"],
            tpl["trajectory_loops_interface"], tpl["trajectory_alpha"], tpl["trajectory_beta"],
            tpl["trajectory_loops"], tpl["trajectory_interface_AA"], trajectory_target_rmsd,
            trajectory_time_text, traj_seq_notes, setting_file,
            tpl["template_mpnn_filters_path"], advanced_file])

    if not advanced_settings["enable_mpnn"]:
        trajectory_n += 1
        continue

    ### MPNN redesign of the interface
    mpnn_n = 0
    accepted_mpnn = 0
    mpnn_dict = {}
    design_start_time = time.time()

    merged_interface_residues = ','.join(sorted(set.union(*interface_residue_sets)))
    first_template = next(iter(templates_run))
    mpnn_trajectories = mpnn_gen_sequence(
        templates_run[first_template]["trajectory_pdb"], binder_chain,
        merged_interface_residues, advanced_settings)

    if advanced_settings["force_reject_AA"]:
        restricted_AAs = set(advanced_settings["omit_AAs"].split(','))
        mpnn_sequences = [
            {'seq': mpnn_trajectories['seq'][n][-length:], 'score': mpnn_trajectories['score'][n],
             'seqid': mpnn_trajectories['seqid'][n]}
            for n in range(advanced_settings["num_seqs"])
            if not any(aa in mpnn_trajectories['seq'][n] for aa in restricted_AAs)]
    else:
        mpnn_sequences = [
            {'seq': mpnn_trajectories['seq'][n][-length:], 'score': mpnn_trajectories['score'][n],
             'seqid': mpnn_trajectories['seqid'][n]}
            for n in range(advanced_settings["num_seqs"])]
    mpnn_sequences.sort(key=lambda x: x['score'])

    if advanced_settings["optimise_beta"] and all(
            float(templates_run[t]['trajectory_beta']) > 15 for t in templates_run):
        advanced_settings["num_recycles_validation"] = advanced_settings["optimise_beta_recycles_valid"]

    ### Compile prediction models once
    clear_mem()
    for template in templates_run:
        tpl = templates_run[template]
        tpl["complex_prediction_model"] = mk_afdesign_model(
            protocol="binder",
            num_recycles=advanced_settings["num_recycles_validation"],
            data_dir=advanced_settings["af_params_dir"],
            use_multimer=multimer_validation,
            use_initial_guess=advanced_settings["predict_initial_guess"],
            use_initial_atom_pos=advanced_settings["predict_bigbang"])

        if advanced_settings["predict_initial_guess"] or advanced_settings["predict_bigbang"]:
            tpl["complex_prediction_model"].prep_inputs(
                pdb_filename=tpl["trajectory_pdb"], chain=general_information["chains"],
                binder_chain="B", binder_len=length, use_binder_template=True,
                rm_target_seq=advanced_settings["rm_template_seq_predict"],
                rm_target_sc=advanced_settings["rm_template_sc_predict"], rm_template_ic=True)
        else:
            tpl["complex_prediction_model"].prep_inputs(
                pdb_filename=tpl["template_pdb"], chain=general_information["chains"],
                binder_len=length,
                rm_target_seq=advanced_settings["rm_template_seq_predict"],
                rm_target_sc=advanced_settings["rm_template_sc_predict"])

    binder_prediction_model = mk_afdesign_model(
        protocol="hallucination", use_templates=False, initial_guess=False,
        use_initial_atom_pos=False,
        num_recycles=advanced_settings["num_recycles_validation"],
        data_dir=advanced_settings["af_params_dir"], use_multimer=multimer_validation)
    binder_prediction_model.prep_inputs(length=length)

    ### Predict and score every MPNN sequence
    for mpnn_sequence in mpnn_sequences:
        mpnn_n += 1
        mpnn_time = time.time()

        if mpnn_sequence['seq'] in [v['seq'] for v in mpnn_dict.values()]:
            print("Skipping duplicate sequence")
            continue

        mpnn_design_name = design_name + "_mpnn" + str(mpnn_n)
        mpnn_score = round(mpnn_sequence['score'], 2)
        mpnn_seqid = round(mpnn_sequence['seqid'], 2)
        mpnn_dict[mpnn_design_name] = {'seq': mpnn_sequence['seq'], 'score': mpnn_score, 'seqid': mpnn_seqid}

        if advanced_settings["save_mpnn_fasta"] is True:
            save_fasta(mpnn_design_name, mpnn_sequence['seq'], design_paths)

        # base AF2 filters, evaluated per template; first template seeds target_statistics
        passed_af2 = True
        target_statistics = None
        for template in templates_run:
            tpl = templates_run[template]
            tpl["mpnn_complex_statistics"], tpl["pass_af2_filters"] = masked_binder_predict_multi_adjusted(
                tpl, mpnn_sequence['seq'], mpnn_design_name, prediction_models,
                advanced_settings, tpl["template_mpnn_filters"], design_paths, target_statistics)
            if target_statistics is None:
                target_statistics = tpl["mpnn_complex_statistics"]
            if not tpl["pass_af2_filters"]:
                print(f"Base AF2 filters not passed for {mpnn_design_name} against "
                      f"{tpl['template_name']}, skipping interface scoring")
                passed_af2 = False
                break

        if not passed_af2:
            continue
        print(f"Base AF2 filters passed for {mpnn_design_name}")

        ### Interface scoring, per template and per model
        for template in templates_run:
            tpl = templates_run[template]
            for model_num in prediction_models:
                mpnn_design_pdb = os.path.join(
                    design_paths["MPNN"], f"{mpnn_design_name}_model{model_num+1}{tpl['template_name']}.pdb")
                mpnn_design_relaxed = os.path.join(
                    design_paths["MPNN/Relaxed"], f"{mpnn_design_name}_model{model_num+1}{tpl['template_name']}.pdb")
                if not os.path.exists(mpnn_design_pdb):
                    continue

                num_clashes_mpnn = calculate_clash_score(mpnn_design_pdb)
                num_clashes_mpnn_relaxed = calculate_clash_score(mpnn_design_relaxed)
                mpnn_interface_scores, mpnn_interface_AA, mpnn_interface_residues = score_interface(
                    mpnn_design_relaxed, binder_chain)
                (mpnn_alpha, mpnn_beta, mpnn_loops, mpnn_alpha_interface, mpnn_beta_interface,
                 mpnn_loops_interface, mpnn_i_plddt, mpnn_ss_plddt) = calc_ss_percentage(
                    mpnn_design_pdb, advanced_settings, binder_chain)
                rmsd_site = unaligned_rmsd(tpl['trajectory_pdb'], mpnn_design_pdb, binder_chain, binder_chain)

                # DEVIATION: multi_binder_design.py:426 discards the division, so Target_RMSD
                # is a sum over the other templates rather than a mean.
                if len(templates_run) > 1:
                    other = [target_pdb_rmsd(mpnn_design_pdb, templates_run[o]["template_pdb"],
                                             general_information["chains"])
                             for o in templates_run if o != template]
                    target_rmsd = sum(other) / len(other)
                else:
                    target_rmsd = np.nan

                # DEVIATION: the script reuses whichever interface residues the last
                # template/model produced; keep this template's own.
                tpl["mpnn_interface_residues"] = mpnn_interface_residues

                tpl["mpnn_complex_statistics"][model_num+1].update({
                    'i_pLDDT': mpnn_i_plddt, 'ss_pLDDT': mpnn_ss_plddt,
                    'Unrelaxed_Clashes': num_clashes_mpnn, 'Relaxed_Clashes': num_clashes_mpnn_relaxed,
                    'Binder_Energy_Score': mpnn_interface_scores['binder_score'],
                    'Surface_Hydrophobicity': mpnn_interface_scores['surface_hydrophobicity'],
                    'ShapeComplementarity': mpnn_interface_scores['interface_sc'],
                    'PackStat': mpnn_interface_scores['interface_packstat'],
                    'dG': mpnn_interface_scores['interface_dG'],
                    'dSASA': mpnn_interface_scores['interface_dSASA'],
                    'dG/dSASA': mpnn_interface_scores['interface_dG_SASA_ratio'],
                    'Interface_SASA_%': mpnn_interface_scores['interface_fraction'],
                    'Interface_Hydrophobicity': mpnn_interface_scores['interface_hydrophobicity'],
                    'n_InterfaceResidues': mpnn_interface_scores['interface_nres'],
                    'n_InterfaceHbonds': mpnn_interface_scores['interface_interface_hbonds'],
                    'InterfaceHbondsPercentage': mpnn_interface_scores['interface_hbond_percentage'],
                    'n_InterfaceUnsatHbonds': mpnn_interface_scores['interface_delta_unsat_hbonds'],
                    'InterfaceUnsatHbondsPercentage': mpnn_interface_scores['interface_delta_unsat_hbonds_percentage'],
                    'InterfaceAAs': mpnn_interface_AA,
                    'Interface_Helix%': mpnn_alpha_interface, 'Interface_BetaSheet%': mpnn_beta_interface,
                    'Interface_Loop%': mpnn_loops_interface, 'Binder_Helix%': mpnn_alpha,
                    'Binder_BetaSheet%': mpnn_beta, 'Binder_Loop%': mpnn_loops,
                    'Hotspot_RMSD': rmsd_site, 'Target_RMSD': target_rmsd})

                if advanced_settings["remove_unrelaxed_complex"]:
                    os.remove(mpnn_design_pdb)

            tpl["mpnn_complex_average_statistics"] = calculate_averages(
                tpl["mpnn_complex_statistics"], handle_aa=True)

        # DEVIATION: the script predicts the binder monomer once per template inside the
        # loop above. It does not depend on the template, so predict it once.
        binder_statistics = predict_binder_alone(
            binder_prediction_model, mpnn_sequence['seq'], mpnn_design_name, length,
            templates_run[first_template]["trajectory_pdb"], binder_chain,
            prediction_models, advanced_settings, design_paths)

        for model_num in prediction_models:
            mpnn_binder_pdb = os.path.join(design_paths["MPNN/Binder"], f"{mpnn_design_name}_model{model_num+1}.pdb")
            # DEVIATION: the script leaves rmsd_binder uninitialised (NameError on a
            # missing first model, stale carry-over afterwards).
            rmsd_binder = None
            if os.path.exists(mpnn_binder_pdb):
                rmsd_binder = unaligned_rmsd(
                    templates_run[first_template]["trajectory_pdb"], mpnn_binder_pdb, binder_chain, "A")
            binder_statistics[model_num+1].update({'Binder_RMSD': rmsd_binder})
            if advanced_settings["remove_binder_monomer"] and os.path.exists(mpnn_binder_pdb):
                os.remove(mpnn_binder_pdb)

        binder_averages = calculate_averages(binder_statistics)

        ### Write one CSV row per template, then apply that template's filters
        mpnn_end_time = time.time() - mpnn_time
        elapsed_mpnn_text = "%d hours, %d minutes, %d seconds" % (
            int(mpnn_end_time // 3600), int((mpnn_end_time % 3600) // 60), int(mpnn_end_time % 60))
        model_numbers = range(1, 6)
        statistics_labels = [
            'pLDDT', 'pTM', 'i_pTM', 'pAE', 'i_pAE', 'i_pLDDT', 'ss_pLDDT', 'Unrelaxed_Clashes',
            'Relaxed_Clashes', 'Binder_Energy_Score', 'Surface_Hydrophobicity', 'ShapeComplementarity',
            'PackStat', 'dG', 'dSASA', 'dG/dSASA', 'Interface_SASA_%', 'Interface_Hydrophobicity',
            'n_InterfaceResidues', 'n_InterfaceHbonds', 'InterfaceHbondsPercentage',
            'n_InterfaceUnsatHbonds', 'InterfaceUnsatHbondsPercentage', 'Interface_Helix%',
            'Interface_BetaSheet%', 'Interface_Loop%', 'Binder_Helix%', 'Binder_BetaSheet%',
            'Binder_Loop%', 'InterfaceAAs', 'Hotspot_RMSD', 'Target_RMSD']

        for template in templates_run:
            tpl = templates_run[template]
            seq_notes = validate_design_sequence(
                mpnn_sequence['seq'],
                tpl["mpnn_complex_average_statistics"].get('Relaxed_Clashes', None),
                advanced_settings)

            mpnn_data = [mpnn_design_name, advanced_settings["design_algorithm"], length, seed,
                         helicity_value, tpl["template_hostspot_residues"], mpnn_sequence['seq'],
                         tpl.get("mpnn_interface_residues", ""), mpnn_score, mpnn_seqid]
            for label in statistics_labels:
                mpnn_data.append(tpl["mpnn_complex_average_statistics"].get(label, None))
                for model in model_numbers:
                    mpnn_data.append(tpl["mpnn_complex_statistics"].get(model, {}).get(label, None))
            for label in ['pLDDT', 'pTM', 'pAE', 'Binder_RMSD']:
                mpnn_data.append(binder_averages.get(label, None))
                for model in model_numbers:
                    mpnn_data.append(binder_statistics.get(model, {}).get(label, None))
            mpnn_data.extend([elapsed_mpnn_text, seq_notes, setting_file,
                              tpl["template_mpnn_filters_path"], advanced_file])

            insert_data(tpl["mpnn_csv"], mpnn_data)
            tpl["mpnn_data"] = mpnn_data

            # DEVIATION: the script scans range(11, 15) and so can never pick model 5.
            plddt_values = {i: mpnn_data[i] for i in range(11, 16) if mpnn_data[i] is not None}
            best_model_number = int(max(plddt_values, key=plddt_values.get)) - 10
            tpl["best_model_pdb"] = os.path.join(
                design_paths["MPNN/Relaxed"],
                f"{mpnn_design_name}_model{best_model_number}{tpl['template_name']}.pdb")

            tpl['filter_conditions'] = check_filters(
                mpnn_data, design_labels, tpl['template_mpnn_filters'])

        all_passed = all(templates_run[t]['filter_conditions'] is True for t in templates_run)

        if all_passed:
            print(f"{mpnn_design_name} passed all filters for all templates")
            accepted_mpnn += 1
            accepted_designs += 1
            for template in templates_run:
                tpl = templates_run[template]
                shutil.copy(tpl["best_model_pdb"], design_paths["Accepted"])
                insert_data(tpl["final_csv"], [''] + tpl["mpnn_data"])

            if advanced_settings["save_design_animations"]:
                for template in templates_run:
                    name = templates_run[template]["template_name"]
                    accepted_animation = os.path.join(
                        design_paths["Accepted/Animation"], f"{design_name}_{name}.html")
                    if not os.path.exists(accepted_animation):
                        shutil.copy(os.path.join(design_paths["Trajectory/Animation"],
                                                 f"{design_name}_{name}.html"), accepted_animation)

            for plot in [f for f in os.listdir(design_paths["Trajectory/Plots"])
                         if f.startswith(design_name) and f.endswith('.png')]:
                target_plot = os.path.join(design_paths["Accepted/Plots"], plot)
                if not os.path.exists(target_plot):
                    shutil.copy(os.path.join(design_paths["Trajectory/Plots"], plot), target_plot)
        else:
            # DEVIATION: the script guards this with `if not filter_conditions:`, which is
            # False for a non-empty list of failures — so it never recorded anything.
            for template in templates_run:
                tpl = templates_run[template]
                conditions = tpl['filter_conditions']
                if conditions is True:
                    continue
                print(f"Unmet filter conditions for {mpnn_design_name} against "
                      f"{tpl['template_name']}: {conditions}")
                failure_df = pd.read_csv(tpl["failure_csv"])
                special_prefixes = ('Average_', '1_', '2_', '3_', '4_', '5_')
                incremented = set()
                for column in conditions:
                    base_column = column
                    for prefix in special_prefixes:
                        if column.startswith(prefix):
                            base_column = column.split('_', 1)[1]
                    if base_column not in incremented and base_column in failure_df.columns:
                        failure_df[base_column] = failure_df[base_column] + 1
                        incremented.add(base_column)
                failure_df.to_csv(tpl["failure_csv"], index=False)
                shutil.copy(tpl["best_model_pdb"], design_paths["Rejected"])

        if accepted_mpnn >= advanced_settings["max_mpnn_sequences"]:
            break

    if accepted_mpnn >= 1:
        print(f"Found {accepted_mpnn} MPNN designs passing filters")
    else:
        print("No accepted MPNN designs found for this trajectory.")
        rejected_designs += 1

    if advanced_settings["remove_unrelaxed_trajectory"]:
        for template in templates_run:
            os.remove(templates_run[template]["trajectory_pdb"])

    design_time = time.time() - design_start_time
    design_time_text = "%d hours, %d minutes, %d seconds" % (
        int(design_time // 3600), int((design_time % 3600) // 60), int(design_time % 60))
    print(f"Design and validation of trajectory {design_name} took: {design_time_text}")

    # rejection-rate monitoring
    if trajectory_n >= advanced_settings["start_monitoring"] and advanced_settings["enable_rejection_check"]:
        if not (accepted_designs / trajectory_n) >= advanced_settings["acceptance_rate"]:
            print("The ratio of successful designs is lower than defined acceptance rate! "
                  "Consider changing your design settings!\nScript execution stopping...")
            break

    trajectory_n += 1

elapsed_time = time.time() - script_start_time
elapsed_text = "%d hours, %d minutes, %d seconds" % (
    int(elapsed_time // 3600), int((elapsed_time % 3600) // 60), int(elapsed_time % 60))
print(f"\nFinished. {trajectory_n} trajectories took: {elapsed_text}")

## 10. Re-predict accepted binders against every template

Output is **long format**, one row per (design, template, model), which is what
seaborn wants:

`Design | Protocol | Seed | Length | Sequence | Predicted_against | Targeting_strategy | Hotspots | Model | pLDDT | pTM | i_pTM | pAE | i_pAE | Design_i_pTM | PDB`

written to `validation_predictions.csv` next to the other CSVs, with the
structures under `Predictions/`. Re-running the cell resumes: rows already in the
CSV are skipped.


In [ ]:
# @title Re-predict accepted binders against every template
# Independent sanity check: take each accepted binder's SEQUENCE ONLY and fold it
# against every template from scratch.
# Output: one row per (design, template, model) long format, ready for seaborn.

from tqdm import tqdm

REPREDICT_DIR = os.path.join(general_information["design_path"], "Predictions")
REPREDICT_CSV = os.path.join(general_information["design_path"], "validation_predictions.csv")
os.makedirs(REPREDICT_DIR, exist_ok=True)

# full target template, unlike the design-time validation which may mask it
RM_TARGET_SEQ = False
RM_TARGET_SC = False
NUM_RECYCLES = advanced_settings["num_recycles_validation"]

### designs to re-predict: accepted ones are identical across templates
### (acceptance requires passing every template), so read one CSV and dedupe.
### "final" = passed every filter; "mpnn" = every MPNN sequence that was scored,
### which is what you want while no design has passed the final filters yet.
DESIGN_SOURCE = "mpnn"          # "final" or "mpnn"

_first_template = template_information[next(iter(template_information))]
first_final_csv = _first_template["final_csv" if DESIGN_SOURCE == "final" else "mpnn_csv"]

# a freshly created CSV is header-only or empty; read_csv raises EmptyDataError on
# the latter, so check the file before trusting it
if not os.path.exists(first_final_csv) or os.path.getsize(first_final_csv) == 0:
    accepted_df = pd.DataFrame()
    print(f"{os.path.basename(first_final_csv)} is missing or empty — nothing to re-predict")
else:
    accepted_df = pd.read_csv(first_final_csv).drop_duplicates(subset="Design")
    print(f"{len(accepted_df)} {DESIGN_SOURCE} design(s) from {os.path.basename(first_final_csv)}")

if accepted_df.empty:
    print("No designs to re-predict yet — run Section 9 until at least one passes.")

### resume: skip (design, template, model) rows already on disk
if os.path.exists(REPREDICT_CSV):
    try:
        predictions_df = pd.read_csv(REPREDICT_CSV)
        done = set(zip(predictions_df["Design"], predictions_df["Predicted_against"], predictions_df["Model"]))
    except pd.errors.EmptyDataError:
        # [FBC-CHANGE] an existing but empty CSV (a prior run with no accepted
        # designs) carries nothing to resume from.
        predictions_df = pd.DataFrame()
        done = set()
    print(f"resuming — {len(done)} prediction(s) already recorded")
else:
    predictions_df = pd.DataFrame()
    done = set()

clear_mem()

# One compiled model per (template, binder length). Rebuilding per design would
# recompile the AF2 graph every time; only a change of binder length forces it.
_model_cache = {}

def _get_prediction_model(template_pdb, binder_len):
    key = (template_pdb, int(binder_len))
    if key not in _model_cache:
        clear_mem()
        _model_cache.clear()
        model = mk_afdesign_model(
            protocol="binder",
            num_recycles=NUM_RECYCLES,
            data_dir=advanced_settings["af_params_dir"],
            use_multimer=multimer_validation,
        )
        model.prep_inputs(
            pdb_filename=template_pdb,
            chain=general_information["chains"],
            binder_len=int(binder_len),
            rm_target_seq=RM_TARGET_SEQ,
            rm_target_sc=RM_TARGET_SC,
        )
        _model_cache[key] = model
    return _model_cache[key]

### re-predict: designs x templates x models
rows = []
work = [(d, t) for _, d in accepted_df.iterrows() for t in template_information]
print(f"{len(work)} design x template pair(s), {len(prediction_models)} model(s) each\n")

for design, template in tqdm(work, desc="Re-predicting", file=sys.stdout):
    tpl = template_information[template]
    template_name = tpl["template_name"]
    design_name = design["Design"]

    binder_sequence = re.sub("[^A-Z]", "", str(design["Sequence"]).upper())
    binder_length = len(binder_sequence)
    if binder_length != int(design["Length"]):
        print(f"  ! {design_name}: CSV Length={design['Length']} but sequence is "
              f"{binder_length} aa — using the sequence")

    model = _get_prediction_model(tpl["template_pdb"], binder_length)

    for model_num in prediction_models:
        if (design_name, template_name, model_num + 1) in done:
            continue

        complex_pdb = os.path.join(
            REPREDICT_DIR, f"{design_name}_vs_{template_name}_model{model_num+1}.pdb")
        model.predict(seq=binder_sequence, models=[model_num],
                      num_recycles=NUM_RECYCLES, verbose=False)
        model.save_pdb(complex_pdb)
        metrics = copy_dict(model.aux["log"])

        rows.append({
            "Design": design_name,
            "Protocol": design["Protocol"],
            "Seed": design["Seed"],
            "Length": binder_length,
            "Sequence": binder_sequence,
            "Predicted_against": template_name,
            "Targeting_strategy": tpl["template_targeting_strategy"],
            "Hotspots": tpl["template_hostspot_residues"],
            "Model": model_num + 1,
            "pLDDT": round(metrics["plddt"], 3),
            "pTM": round(metrics["ptm"], 3),
            "i_pTM": round(metrics["i_ptm"], 3),
            "pAE": round(metrics["pae"], 3),
            "i_pAE": round(metrics["i_pae"], 3),
            "Design_i_pTM": design.get("Average_i_pTM", np.nan),   # design-stage value, for comparison
            "PDB": complex_pdb,
        })

if rows:
    predictions_df = pd.concat([predictions_df, pd.DataFrame(rows)], ignore_index=True)
predictions_df.to_csv(REPREDICT_CSV, index=False)
print(f"\nwrote {len(predictions_df)} row(s) -> {REPREDICT_CSV}")
print(f"structures            -> {REPREDICT_DIR}")

### quick look: mean i_pTM per design per template
if not predictions_df.empty:
    pivot = predictions_df.pivot_table(index="Design", columns="Predicted_against",
                                       values="i_pTM", aggfunc="mean").round(3)
    print("\nmean i_pTM (re-predicted, models averaged):")
    print(pivot.to_string())

## 11. Compare the templates

The figure is saved next to the CSVs as `validation_metrics.png`, and the same
numbers are printed underneath as a table, so nothing is readable only as colour.

In [ ]:
# @title Violin comparison of re-predicted metrics
# One panel per metric, templates on the x axis. Separate panels rather than one
# shared axis on purpose: i_pTM lives in 0-1 and PAE in ~0-30, and putting two
# scales on one plot invents a relationship that is not in the data.
import matplotlib.pyplot as plt

# reference palette, light mode. Slots 1-3 are the documented all-pairs-validated
# set; small multiples are an all-pairs form, so past three templates identity
# goes to the x axis alone rather than to a generated fourth hue.
SERIES  = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"

PLOT_METRICS = [
    ("i_pTM", "interface pTM",  "higher is better"),
    ("pLDDT", "pLDDT",          "higher is better"),
    ("pTM",   "pTM",            "higher is better"),
    ("i_pAE", "interface PAE",  "lower is better"),
    ("pAE",   "PAE",            "lower is better"),
]

try:
    pred_df = pd.read_csv(REPREDICT_CSV)
except (FileNotFoundError, pd.errors.EmptyDataError):
    # [FBC-CHANGE] Section 10 writes an empty CSV when there is nothing to
    # re-predict (e.g. a 1-trajectory run that accepted no designs); treat it
    # as "no data" instead of crashing on read.
    pred_df = pd.DataFrame()

if not pred_df.empty and "Predicted_against" in pred_df.columns:
    order = [template_information[t]["template_name"] for t in template_information]
    order = [t for t in order if t in set(pred_df["Predicted_against"])]
else:
    order = []

if pred_df.empty or not order:
    print("No re-predictions yet — run Section 10 first.")
else:
    counts = pred_df.groupby("Predicted_against").size().to_dict()
    n_min = min(counts[t] for t in order)
    # a KDE drawn over a handful of points is invention, not a distribution
    use_violin = n_min >= 5
    colors = SERIES[:len(order)] if len(order) <= 3 else [SERIES[0]] * len(order)

    ncols = min(3, len(PLOT_METRICS))
    nrows = -(-len(PLOT_METRICS) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.3 * ncols, 3.7 * nrows),
                             facecolor=SURFACE)
    axes = np.atleast_1d(axes).ravel()

    for ax, (metric, label, direction) in zip(axes, PLOT_METRICS):
        ax.set_facecolor(SURFACE)
        groups = [pred_df.loc[pred_df["Predicted_against"] == t, metric].dropna().values
                  for t in order]

        if use_violin:
            parts = ax.violinplot(groups, positions=range(len(order)), widths=0.68,
                                  showmeans=False, showmedians=False, showextrema=False)
            for body, vals, c in zip(parts["bodies"], groups, colors):
                # clip the KDE to the observed range: no tails into impossible values
                v = body.get_paths()[0].vertices
                v[:, 1] = np.clip(v[:, 1], vals.min(), vals.max())
                body.set_facecolor(c); body.set_alpha(0.30)
                body.set_edgecolor(c); body.set_linewidth(1.0)

        for i, (vals, c) in enumerate(zip(groups, colors)):
            if not len(vals):
                continue
            jitter = (np.random.default_rng(1000 + i).random(len(vals)) - 0.5) * 0.22
            ax.scatter(np.full(len(vals), i) + jitter, vals, s=26, color=c,
                       edgecolor=SURFACE, linewidth=0.9, alpha=0.95, zorder=3)
            med = float(np.median(vals))
            ax.hlines(med, i - 0.2, i + 0.2, color=INK, linewidth=1.6, zorder=4)

        ax.set_title(f"{label}", color=INK, fontsize=11, pad=24, loc="left")
        ax.text(0, 1.008, direction, transform=ax.transAxes, color=MUTED,
                fontsize=8.5, va="bottom")
        ax.set_xticks(range(len(order)))
        ax.set_xticklabels([f"{t}\nn={counts[t]}" for t in order], color=INK2, fontsize=9.5)
        ax.tick_params(axis="y", colors=MUTED, labelsize=9, length=0)
        ax.tick_params(axis="x", length=0)
        ax.set_axisbelow(True)
        ax.grid(axis="y", color=GRID, linewidth=0.8, linestyle="-")   # solid hairline
        for side in ("top", "right"):
            ax.spines[side].set_visible(False)
        for side in ("left", "bottom"):
            ax.spines[side].set_color(AXIS); ax.spines[side].set_linewidth(0.8)

    for ax in axes[len(PLOT_METRICS):]:
        ax.set_visible(False)

    kind = "violin + points" if use_violin else "points only (n too small for a density)"
    fig.suptitle(f"Re-predicted metrics per template  :  {kind}",
                 color=INK, fontsize=13, x=0.005, ha="left", y=0.995)
    fig.tight_layout(rect=(0, 0, 1, 0.97))

    PLOT_PNG = os.path.join(general_information["design_path"], "validation_metrics.png")
    fig.savefig(PLOT_PNG, dpi=200, facecolor=SURFACE, bbox_inches="tight")
    plt.show()
    print(f"saved {PLOT_PNG}")

    # table twin: every value in the figure is also readable as numbers
    summary = (pred_df.groupby("Predicted_against")[[m for m, _, _ in PLOT_METRICS]]
                      .agg(["median", "min", "max"]).round(3))
    summary.insert(0, "n", pd.Series(counts))
    print("\n" + summary.to_string())